# ROGII — OOF Correlation & Blend + **visible-prefix router probe** (k=3 refeed, cache-first)

Standalone Colab notebook. Reuses the **training** data/model/eval machinery verbatim — env setup, deterministic `fold_splits`, the real-image cache, `RealValDataset`, `build_model`, and the decode. Takes a list of checkpoint dirs (one per member) and scores them out-of-fold.

**Fixed inference contract:** every member is evaluated at **k=3 self-refeed, `REFEED_TRAIN_T = HANDOFF_TRAIN_T = 1.0`**. There is no depth sweep — the depth is pinned and asserted.

**Cache-first driver.** If a member's dir already holds the per-seed exports written by the training notebook's §C.2 cell (`oof_f{fold}_s{seed}.npz`), the driver **loads them instead of running inference**. The seed-ensemble is reconstructed exactly: the decode `er -> pred_col -> pred_row` is affine in `er`, so averaging the per-seed *expected rows* (what `_eval_model_pool` does) and averaging the per-seed *residuals* (what the cache gives) are the same number. Checkpoints are only needed when the cache is absent or its provenance doesn't match (k=3 / T=1.0).

**Contents:** per-member OOF RMSE + residual correlation (incl. worst decile) → global weight blend → datum/drift + toe decomposition → **§13 visible-prefix holdout router probe** (idea #1).


### 1 · Colab mount

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


### 2 · Environment / data root

In [ ]:
import os, zipfile
ZIP_PATH = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/rogii-wellbore-geology-prediction.zip'
UNZIP_DIR = '/content/rogii'
if IN_COLAB:
    if os.path.exists(UNZIP_DIR) and len(os.listdir(UNZIP_DIR)) > 0:
        print(f'Dataset already at {UNZIP_DIR}.')
    else:
        os.makedirs(UNZIP_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf: zf.extractall(UNZIP_DIR)
    DATA_ROOT = UNZIP_DIR
else:
    DATA_ROOT = '/path/to/rogii-wellbore-geology-prediction'

# === Architecture constants — identical across all members; ENCODER is overridden per member in run_oof ===
ENCODER          = 'efficientnet-b7'          # default/fallback (the OOF driver sets the real one per member)
DECODER_CHANNELS = (512, 256, 128, 64, 32)    # 2x decoder, shared by every member
DECODER_INTERP   = 'nearest'                   # native SMP interp (no pixelshuffle)
STACK_N          = 2                            # 2-stage stacked UNet (stage-2 = headline)
STACK_DETACH     = True                         # detach stage-1 posterior before stage-2
print(f'Arch: STACK_N={STACK_N}, detach={STACK_DETACH}, decoder={DECODER_CHANNELS}, interp={DECODER_INTERP}')


### 3 · Imports + locked constants

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation_models_pytorch'])
    import segmentation_models_pytorch as smp
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm
print(f'smp {smp.__version__}, timm {timm.__version__}')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

IN_CH     = 5            # 4 baseline channels + dZ-per-column
EMA_DECAY = 0.997        # used by the EMAState definition (checkpoints were trained with EMA)
SEED      = 34
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}' + (f'  GPU: {torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))

# === Determinism (deterministic eval) ===
import random as _random, os as _os
_os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    """worker_init_fn for DataLoader — reseeds numpy + python random per worker."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); _random.seed(worker_seed)
print('Imports + determinism ready.')


### 4 · Deterministic fold splits

In [ ]:
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
HW_SUFFIX = '__horizontal_well.csv'
TW_SUFFIX = '__typewell.csv'
def well_hw_path(wid): return os.path.join(TRAIN_DIR, wid + HW_SUFFIX)
def well_tw_path(wid): return os.path.join(TRAIN_DIR, wid + TW_SUFFIX)
def list_wells(wells_dir):
    files = set(os.listdir(wells_dir))
    return [f[:-len(HW_SUFFIX)] for f in sorted(files)
            if f.endswith(HW_SUFFIX) and (f[:-len(HW_SUFFIX)] + TW_SUFFIX) in files]
well_ids_all = list_wells(TRAIN_DIR)
print(f'{len(well_ids_all)} wells.')

rng = np.random.RandomState(SEED)
well_pool = list(well_ids_all); rng.shuffle(well_pool)
well_meta = []
for wid in tqdm(well_pool, desc='scanning'):
    hw = pd.read_csv(well_hw_path(wid), usecols=['MD','TVT','TVT_input'])
    well_meta.append({'well_id': wid, 'n_rows': len(hw),
                      'tvt_range': hw['TVT'].max() - hw['TVT'].min()})
meta_df = pd.DataFrame(well_meta)
N_STRATA = N_FOLDS = 5
meta_df['stratum'] = pd.qcut(meta_df['tvt_range'], q=N_STRATA, labels=list(range(N_STRATA)))
fold_assignment = {}
for s in range(N_STRATA):
    pool = meta_df[meta_df['stratum'] == s]['well_id'].tolist()
    rng2 = np.random.RandomState(SEED + s); rng2.shuffle(pool)
    chunk = len(pool) // N_FOLDS
    for f in range(N_FOLDS):
        lo = f * chunk; hi = (f + 1) * chunk if f < N_FOLDS - 1 else len(pool)
        for wid in pool[lo:hi]: fold_assignment[wid] = f
fold_splits = {}
for f in range(N_FOLDS):
    fold_splits[f] = {
        'val':   [w for w, fa in fold_assignment.items() if fa == f],
        'train': [w for w, fa in fold_assignment.items() if fa != f],
    }
    print(f'Fold {f}: {len(fold_splits[f]["train"])} train, {len(fold_splits[f]["val"])} val')

# === FT26: full 5-fold sweep (no probe restriction) ===
ALL_FOLDS = [0, 1, 2, 3, 4]
print(f'\nFT26 using all {len(ALL_FOLDS)} folds: {ALL_FOLDS}')


### 4b · OOF over all five folds

In [ ]:
# OOF spans ALL five folds (the training probe had ALL_FOLDS=[1]); fold_splits holds all 5.
ALL_FOLDS = [0, 1, 2, 3, 4]
print('OOF folds:', ALL_FOLDS)


### 5 · Real-image cache (5-ch incl. dZ)


In [ ]:
H = 256; W = 512; BIN_SIZE = 1.0; K_MAX = 12
#H = 224; W = 448; BIN_SIZE = 256/224; K_MAX = 12


# === dZ-per-column channel helpers (FT39e) ===
def _dz_per_col(hw_z, hw_md, bin_edges, hw_mask):
    """Per-column average of dZ/dMD. Returns float32 (W,) with NaN where no data."""
    n = len(hw_z)
    if n < 2:
        return np.full(W, 0.0, dtype=np.float32)
    # uniform 1.0 ft MD step per the EDA report, but compute defensively
    dz_per_row = np.gradient(np.asarray(hw_z, dtype=np.float64),
                              np.asarray(hw_md, dtype=np.float64))
    dz = np.full(W, np.nan, dtype=np.float32)
    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo:
            continue
        dz[c] = float(np.mean(dz_per_row[lo:hi]))
    return dz

def _dz_channel(dz_per_col, hw_mask):
    """Build the (H, W) dZ channel — per-well z-score + clip ±3 + broadcast across H.
    Neutralize columns without HW data so they don't bias the well z-score / clip."""
    bad = (~hw_mask) | (~np.isfinite(dz_per_col))
    if (~bad).any():
        fill = float(np.nanmean(dz_per_col[~bad]))
    else:
        fill = 0.0
    dz = np.where(bad, fill, dz_per_col).astype(np.float32)
    m, s = dz.mean(), dz.std()
    dz_norm = (dz - m) / s if s > 1e-6 else np.zeros_like(dz)
    dz_norm = np.clip(dz_norm, -3.0, 3.0)
    return np.broadcast_to(dz_norm[None, :], (H, W)).copy().astype(np.float32)

def build_image_from_arrays(hw_md, hw_gr, hw_z, hw_tvt, hw_tvti, tw_tvt, tw_gr,
                            phase_h=0.0, phase_v=0.0):
    n_hw = len(hw_md)
    known_tvt = pd.Series(hw_tvti).dropna().values
    if len(known_tvt) == 0: raise ValueError('No pre-PS rows')
    depth_center = float(known_tvt[-1])
    tvt_lo = depth_center - (H/2) * BIN_SIZE
    if phase_v != 0.0:                       # sub-bin TVT grid phase (PhaseJitter)
        tvt_lo = tvt_lo + float(phase_v) * BIN_SIZE
    def tvt_to_row(t, lo=tvt_lo):
        if np.isnan(t): return -1
        r = int((t - lo) / BIN_SIZE)
        return r if 0 <= r < H else -1
    if phase_h != 0.0:                       # sub-bin MD grid phase (PhaseJitter)
        bin_edges = np.clip(np.linspace(0, n_hw, W + 1)
                            + float(phase_h) * (n_hw / W), 0, n_hw).astype(int)
    else:
        bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
    hw_gr_per_col_raw = np.full(W, np.nan, dtype=np.float32)
    hw_mask = np.zeros(W, dtype=bool)
    truth_row = np.full(W, -1, dtype=np.int32)
    hint_row = np.full(W, -1, dtype=np.int32)
    post_ps_mask = np.zeros(W, dtype=bool)
    for c in range(W):
        lo, hi = bin_edges[c], bin_edges[c+1]
        if hi <= lo: continue
        gr_slc = hw_gr[lo:hi]
        if np.any(~np.isnan(gr_slc)):
            hw_gr_per_col_raw[c] = np.nanmean(gr_slc)
            hw_mask[c] = True
        tvt_slc = hw_tvt[lo:hi]
        if np.any(~np.isnan(tvt_slc)): truth_row[c] = tvt_to_row(np.nanmedian(tvt_slc))
        tvti_slc = hw_tvti[lo:hi]
        is_known = ~np.isnan(tvti_slc)
        if is_known.any(): hint_row[c] = tvt_to_row(np.nanmedian(tvti_slc[is_known]))
        else: post_ps_mask[c] = True
    if np.all(np.isnan(hw_gr_per_col_raw)):
        hw_gr_per_col = np.zeros(W, dtype=np.float32)
    else:
        col_mean = np.nanmean(hw_gr_per_col_raw)
        hw_gr_per_col = hw_gr_per_col_raw.copy()
        hw_gr_per_col[np.isnan(hw_gr_per_col)] = col_mean
    ext_row_tvts = (tvt_lo - K_MAX * BIN_SIZE) + (np.arange(H + 2*K_MAX) + 0.5) * BIN_SIZE
    sort_idx = np.argsort(tw_tvt)
    tw_tvt_s = tw_tvt[sort_idx]; tw_gr_s = tw_gr[sort_idx]
    ok = ~np.isnan(tw_gr_s)
    if ok.sum() < 2:
        tw_gr_per_row_ext = np.zeros(H + 2*K_MAX, dtype=np.float32)
    else:
        tw_gr_per_row_ext = np.interp(ext_row_tvts, tw_tvt_s[ok], tw_gr_s[ok],
                                       left=tw_gr_s[ok][0], right=tw_gr_s[ok][-1]).astype(np.float32)
    tw_gr_per_row = tw_gr_per_row_ext[K_MAX : K_MAX + H]
    ch_tw_raw = np.broadcast_to(tw_gr_per_row[:, None], (H, W)).copy()
    ch_hw_raw = np.broadcast_to(hw_gr_per_col[None, :], (H, W)).copy()
    ch_hint = np.zeros((H, W), dtype=np.float32)
    for c in range(W):
        if hint_row[c] >= 0 and not post_ps_mask[c]:
            ch_hint[hint_row[c], c] = 1.0
    cost_vol = (ch_hw_raw - ch_tw_raw) ** 2
    if hw_mask.any():
        mean_cost = float(cost_vol[:, hw_mask].mean())
    else:
        mean_cost = 0.0
    for c in range(W):
        if not hw_mask[c]:
            cost_vol[:, c] = mean_cost
    ch_tw = ch_tw_raw.copy()
    m, s = ch_tw.mean(), ch_tw.std()
    if s > 1e-6: ch_tw = (ch_tw - m) / s
    ch_hw = ch_hw_raw.copy()
    m, s = ch_hw.mean(), ch_hw.std()
    if s > 1e-6: ch_hw = (ch_hw - m) / s
    cm, cs = cost_vol.mean(), cost_vol.std()
    if cs > 1e-6:
        cost_vol_norm = (cost_vol - cm) / cs
    else:
        cost_vol_norm = np.zeros_like(cost_vol)
    cost_vol_norm = np.clip(cost_vol_norm, -3.0, 3.0)
    # --- dZ-per-column channel (FT39e) — jitter-invariant (per-column scalar, no row shift) ---
    dz_per_col = _dz_per_col(hw_z, hw_md, bin_edges, hw_mask)
    ch_dz = _dz_channel(dz_per_col, hw_mask)
    image_in = np.stack([ch_tw, ch_hw, ch_hint, cost_vol_norm, ch_dz], axis=0).astype(np.float32)
    return dict(image_in=image_in, truth_row=truth_row, hint_row=hint_row,
                post_ps_mask=post_ps_mask, depth_center=depth_center,
                tvt_lo=tvt_lo, hw_mask=hw_mask, dz_per_col=dz_per_col,
                tw_gr_per_row_ext=tw_gr_per_row_ext, hw_gr_per_col=hw_gr_per_col)

def build_image_real(hw_df, tw_df):
    arrs = dict(
        hw_md=hw_df['MD'].values, hw_gr=hw_df['GR'].values,
        hw_z=hw_df['Z'].values,
        hw_tvt=hw_df['TVT'].values, hw_tvti=hw_df['TVT_input'].values,
        tw_tvt=tw_df['TVT'].values, tw_gr=tw_df['GR'].values)
    rec = build_image_from_arrays(**arrs)
    # Raw logs retained EXACTLY as passed (no dtype cast) so render_phase(rec, 0, 0)
    # reproduces the cached build bit-for-bit. Real wells only (~200 MB total);
    # synth recs never carry raws and are never phase-jittered.
    rec['_raw'] = arrs
    return rec


# Build real images for all 773 wells (every well appears in train of 4 folds, val of 1)
print(f'Building all {len(well_ids_all)} real images...')
well_images_real = {}
for wid in tqdm(sorted(well_ids_all), desc='real'):
    hw = pd.read_csv(well_hw_path(wid))
    tw = pd.read_csv(well_tw_path(wid))
    rec = build_image_real(hw, tw)
    rec['well_id'] = wid; rec['_is_synth'] = False
    well_images_real[wid] = rec
print(f'Built {len(well_images_real)} real images')


### 6 · RealValDataset (canonical cached image)

In [ ]:
class RealValDataset(Dataset):
    """Val dataset. Both arms use sparse hint here — no arm-conditional behavior needed.
    (FT21 bug class cannot apply by construction.)
    Uses cached image_in (5ch incl. dZ) from cell 12."""
    def __init__(self, well_ids):
        self.well_ids = well_ids
    def __len__(self): return len(self.well_ids)
    def __getitem__(self, idx):
        wid = self.well_ids[idx]
        rec = well_images_real[wid]
        image = rec['image_in']
        truth_row = rec['truth_row']
        post_ps = rec['post_ps_mask']
        main_mask = post_ps & (truth_row >= 0)
        truth_safe = np.where(main_mask, truth_row, 0).astype(np.int64)
        return {'image': torch.from_numpy(image).clone(),
                'truth_row': torch.from_numpy(truth_safe).long(),
                'main_mask': torch.from_numpy(main_mask).clone(),
                'well_id': wid}


### 7 · Model + build_model + fold_const


In [ ]:
DROP_CONNECT_RATE = None
DROP_PATH_RATE = None

# === Iterative-refinement (k-refeed) inference temperatures =================================
# The stage-1->stage-2 handoff and the stage-2 self-refeed can each be softened by a temperature.
# BOTH default to 1.0 == plain softmax, which makes k=1 inference BIT-IDENTICAL to the original
# single-pass swap notebook. Raise >1 only if a member was trained to consume a softened belief
# (these backbone-swap checkpoints were not, so 1.0 is the faithful default).
HANDOFF_TRAIN_T = 1.0   # stage-1 -> stage-2 handoff temperature (applies at every k>=1)
REFEED_TRAIN_T  = 1.0   # stage-2 self-refeed temperature (applies on refeed rounds 2..k)

def _apply_drop_connect(unet, rate):
    """Override SMP native-EfficientNet stochastic-depth (drop_connect) rate in place.
    rate=None -> untouched (bit-exact). No-op (with warning) for encoders that don't expose it
    (resnet/timm 'tu-' encoders set drop_path_rate at build, not post-hoc)."""
    if rate is None:
        return
    enc = unet.encoder
    if hasattr(enc, '_drop_connect_rate'):
        enc._drop_connect_rate = float(rate)
    else:
        import warnings
        warnings.warn(f"{type(enc).__name__} has no _drop_connect_rate; "
                      f"DROP_CONNECT_RATE={rate} ignored (native smp efficientnet-bX only).")


class UNetGeneric(nn.Module):
    def __init__(self, encoder_name='resnet34', in_ch=5):
        super().__init__()
        self.smp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                            in_channels=in_ch, classes=1,
                            decoder_channels=DECODER_CHANNELS,
                            decoder_interpolation=_smp_interp_mode(DECODER_INTERP))
        _apply_drop_connect(self.smp, DROP_CONNECT_RATE)
    def forward(self, x): return self.smp(x).squeeze(1)

def _assert_droppath_active(unet, rate, tag):
    """Fail LOUD if drop_path_rate was requested but the backbone built NO active
    stochastic-depth layers. timm HRNet swallows drop_path_rate via **kwargs->cfg.update
    and applies nothing (silent no-op); efficientnet/resnet/convnext/regnet honor it."""
    if not rate:
        return
    from timm.layers import DropPath
    live = [getattr(m, "drop_prob", 0.0) for m in unet.modules() if isinstance(m, DropPath)]
    live = [p for p in live if p and p > 0.0]
    assert live, (
        f"drop_path_rate={rate} requested but {tag} built 0 active DropPath layers -> "
        f"this backbone IGNORES it (silent no-op). Wire stochastic depth explicitly, or use "
        f"a backbone that honors drop_path (efficientnet/resnet/convnext/regnet).")


def _smp_interp_mode(m):
    return m   # native SMP interpolation


class StackedUNet(nn.Module):
    def __init__(self, encoder_name='resnet34', in_ch=5, detach_handoff=False):
        super().__init__()
        self.detach_handoff = detach_handoff
        self.stage1 = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                               in_channels=in_ch, classes=1,
                               decoder_channels=DECODER_CHANNELS,
                               decoder_interpolation=_smp_interp_mode(DECODER_INTERP),
                               drop_path_rate=DROP_PATH_RATE)
        self.stage2 = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                               in_channels=in_ch + 1, classes=1,
                               decoder_channels=DECODER_CHANNELS,
                               decoder_interpolation=_smp_interp_mode(DECODER_INTERP),
                               drop_path_rate=DROP_PATH_RATE)
        _apply_drop_connect(self.stage1, DROP_CONNECT_RATE)   # set on BOTH encoders
        _apply_drop_connect(self.stage2, DROP_CONNECT_RATE)
        _assert_droppath_active(self.stage1, DROP_PATH_RATE, f"{encoder_name} stage1")
        _assert_droppath_active(self.stage2, DROP_PATH_RATE, f"{encoder_name} stage2")

    def forward(self, x, return_aux=False):
        logits1 = self.stage1(x).squeeze(1)                 # (B, H, W)
        handoff = logits1.detach() if self.detach_handoff else logits1
        post1   = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)  # T=1.0 -> identical to plain softmax
        x2      = torch.cat([x, post1.unsqueeze(1)], dim=1) # (B, IN_CH+1, H, W)
        logits2 = self.stage2(x2).squeeze(1)                # (B, H, W)
        if return_aux:
            return logits2, logits1                          # logits1 ATTACHED -> deep-sup CE1
        return logits2

    def _stage2_from_post(self, x, post):
        """One stage-2 pass given the input image x and a row-posterior `post` (B,H,W). Mirrors
        forward()'s stage-2 handling so a self-refeed round is identical to the first round except
        for which belief is fed. Returns stage-2 logits (B,H,W)."""
        x2 = torch.cat([x, post.unsqueeze(1)], dim=1)
        return self.stage2(x2).squeeze(1)

def build_model(encoder_name=None, in_ch=None):
    """Single source of truth for model construction (training AND the eval/OOF
    reconstruction in section C). STACK_N>=2 -> StackedUNet, else the bit-exact single
    UNetGeneric. Reads the globals at call time so one STACK_N toggle switches every
    reconstruction site consistently."""
    enc = ENCODER if encoder_name is None else encoder_name
    ich = IN_CH   if in_ch       is None else in_ch
    if STACK_N >= 2:
        return StackedUNet(encoder_name=enc, in_ch=ich, detach_handoff=STACK_DETACH)
    return UNetGeneric(encoder_name=enc, in_ch=ich)

def hard_ce_loss(logits, truth_row, mask):
    B, Hh, Ww = logits.shape
    log_probs = F.log_softmax(logits, dim=1).permute(0, 2, 1).reshape(B*Ww, Hh)
    tgt = truth_row.reshape(B*Ww); msk = mask.reshape(B*Ww)
    nll = -log_probs.gather(1, tgt.unsqueeze(1)).squeeze(1)
    return (nll * msk.float()).sum() / msk.float().sum().clamp(min=1.0)

def exp_row_loss(logits, truth_row, mask, delta=8.0):
    """Integral-regression term: Huber between the softmax-EXPECTED row (the eval
    decode) and the true row, over valid columns. Same quantity the metric scores."""
    B, Hh, Ww = logits.shape
    rows = torch.arange(Hh, device=logits.device, dtype=logits.dtype)   # (H,)
    probs = F.softmax(logits, dim=1)                                    # (B,H,W)
    er = (probs * rows[None, :, None]).sum(dim=1)                       # (B,W) expected row
    e  = (er - truth_row.to(logits.dtype)).abs()                       # (B,W)
    huber = torch.where(e < delta, 0.5 * e**2, delta * (e - 0.5 * delta))
    m = mask.to(logits.dtype)
    return (huber * m).sum() / m.sum().clamp(min=1.0)

def constant_baseline_pooled(val_ids):
    all_errs_sq = []
    for wid in val_ids:
        hw = pd.read_csv(well_hw_path(wid), usecols=['TVT_input','TVT'])
        post = hw['TVT_input'].isna().values
        if post.sum() == 0: continue
        last_known = hw['TVT_input'].dropna().iloc[-1]
        true_post = hw['TVT'].values[post]
        errs = true_post - last_known
        all_errs_sq.append(errs**2)
    return float(np.sqrt(np.concatenate(all_errs_sq).mean()))

fold_const = {f: constant_baseline_pooled(fold_splits[f]['val']) for f in ALL_FOLDS}
for f in ALL_FOLDS:
    print(f'Fold {f} constant: {fold_const[f]:.3f} ft')


# === EMA state (lifted from FT17c) ===
class EMAState:
    """Exponential moving average of model parameters.
    Tracks all float-dtype state_dict entries (params + BN running stats).
    """
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point:
                    self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
    def swap_into(self, model):
        live = {}
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in self.shadow:
                    live[k] = v.detach().clone()
                    v.copy_(self.shadow[k])
        return live
    def restore(self, model, live):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in live:
                    v.copy_(live[k])

def make_lr_lambda(warmup_epochs, total_epochs):
    """Linear warmup then cosine."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * min(1.0, max(0.0, progress))))
    return lr_lambda

def make_lr_lambda_warm_restarts(warmup_epochs, total_epochs, T_0, T_mult):
    """Linear warmup, then SGDR-style cosine annealing with warm restarts.

    After warmup, divides remaining epochs into restart cycles of length
    T_0, T_0*T_mult, T_0*T_mult^2, ...  At the start of each cycle the LR
    jumps back to its max value, then cosine-anneals to ~0 over that cycle.

    Examples (T_0=10):
      T_mult=1: cycles at ep [0-10, 10-20, 20-30, 30-40] (4 equal kicks)
      T_mult=2: cycles at ep [0-10, 10-30, 30-70]        (1 restart, then long anneal)
    """
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        e = epoch - warmup_epochs
        cycle_start = 0; T_cur = T_0
        while cycle_start + T_cur <= e:
            cycle_start += T_cur
            T_cur *= T_mult
        progress = (e - cycle_start) / T_cur
        return 0.5 * (1.0 + np.cos(np.pi * min(1.0, max(0.0, progress))))
    return lr_lambda

def warm_restart_cycle_epochs(warmup_epochs, total_epochs, T_0, T_mult):
    """Absolute (0-based) epoch indices that BEGIN a cycle (restarts) and that
    END a cycle (low-LR settle points), matching make_lr_lambda_warm_restarts."""
    restarts, ends = set(), set()
    n = total_epochs - warmup_epochs
    cs, T = 0, T_0
    while cs < n:
        restarts.add(warmup_epochs + cs)
        ends.add(warmup_epochs + min(cs + T - 1, n - 1))
        cs += T; T *= T_mult
    return restarts, ends

### 8 · Decode + evaluate_model

In [ ]:
row_idx_t = torch.arange(H, device=DEVICE, dtype=torch.float32)

def _decode_expected_err(er_b_np, rec, hw_cache):
    """Decode one well's per-column expected row -> per-row post-PS squared errors.
    Shared by stage-2 and stage-1 so both decode identically (single source of truth)."""
    n_hw, bin_edges, post, true_post, ok = hw_cache
    pred_col = rec['tvt_lo'] + (er_b_np + 0.5) * BIN_SIZE
    pred_full = np.zeros(n_hw, dtype=np.float32)
    for c in range(W): pred_full[bin_edges[c]:bin_edges[c+1]] = pred_col[c]
    return (true_post[ok] - pred_full[post][ok])**2


def _eval_iter_logits(model, img, kmax):
    """Eval-time self-refeed iteration MATCHING DEPLOYMENT: stage-1 -> handoff (softened by
    HANDOFF_TRAIN_T, same path as forward) -> stage-2 (k=1), then re-feed stage-2's OWN softmax
    posterior (softened by REFEED_TRAIN_T) for rounds 2..kmax. Returns ({k: logits2 for k=1..kmax},
    logits1). No grad. k=1 is bit-identical to model.forward() when the temps are 1.0. Uses the
    model's own _stage2_from_post so decode == forward at k=1."""
    logits1 = model.stage1(img).squeeze(1)
    handoff = logits1.detach() if model.detach_handoff else logits1
    post = F.softmax(handoff / HANDOFF_TRAIN_T, dim=1)
    out = {}
    logits2 = model._stage2_from_post(img, post)
    out[1] = logits2
    for k in range(2, kmax + 1):
        post = F.softmax(logits2 / REFEED_TRAIN_T, dim=1)   # re-feed stage-2's own belief (deploy==train)
        logits2 = model._stage2_from_post(img, post)
        out[k] = logits2
    return out, logits1


def evaluate_model(model, loader, k_list=None):
    """Eval -> softmax-expected row -> post-PS TVT-RMSE + masked val CE (stage-2 = headline).

    For the stacked model (STACK_N>=2) ALSO decode the STAGE-1 posterior in the SAME pass and
    return its standalone TVT-RMSE + CE (base vs refinement split).

    Returns (rmse2, ce2, rmse1, ce1) as before -> the first two are the k=1 stage-2 headline. If
    k_list is given (e.g. [2,3,4]) and stacked, a 5th return `multik` is a dict {k: (rmse_k, ce_k)}
    for each requested depth, using the deployment-matched self-refeed. k_list is observational.
    """
    model.eval()
    stacked = (STACK_N >= 2)
    want_k = sorted(set(int(k) for k in (k_list or []) if int(k) >= 2)) if stacked else []
    kmax = max([1] + want_k)
    errs2, errs1 = [], []
    errs_k = {k: [] for k in want_k}
    loss2_sum, loss1_sum, n_batches = 0.0, 0.0, 0
    lossk_sum = {k: 0.0 for k in want_k}
    with torch.no_grad():
        for batch in loader:
            img = batch['image'].to(DEVICE)
            val_tgt = batch['truth_row'].to(DEVICE); val_msk = batch['main_mask'].to(DEVICE)
            if want_k:
                kl, logits1 = _eval_iter_logits(model, img, kmax)
                logits = kl[1]                                   # k=1 stage-2 == forward() headline
            elif stacked:
                logits, logits1 = model(img, return_aux=True)
            else:
                logits, logits1 = model(img), None
            loss2_sum += hard_ce_loss(logits, val_tgt, val_msk).item()
            er2 = (F.softmax(logits, dim=1) * row_idx_t[None, :, None]).sum(dim=1)
            if stacked:
                loss1_sum += hard_ce_loss(logits1, val_tgt, val_msk).item()
                er1 = (F.softmax(logits1, dim=1) * row_idx_t[None, :, None]).sum(dim=1)
            for k in want_k:
                lossk_sum[k] += hard_ce_loss(kl[k], val_tgt, val_msk).item()
            n_batches += 1
            for b in range(img.size(0)):
                wid = batch['well_id'][b]; rec = well_images_real[wid]
                hw = pd.read_csv(well_hw_path(wid), usecols=['TVT','TVT_input'])
                n_hw = len(hw); bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
                post = hw['TVT_input'].isna().values
                true_post = hw['TVT'].values[post]; ok = ~np.isnan(true_post)
                hw_cache = (n_hw, bin_edges, post, true_post, ok)
                errs2.append(_decode_expected_err(er2[b].cpu().numpy(), rec, hw_cache))
                if stacked:
                    errs1.append(_decode_expected_err(er1[b].cpu().numpy(), rec, hw_cache))
                for k in want_k:
                    er_k = (F.softmax(kl[k], dim=1) * row_idx_t[None, :, None]).sum(dim=1)
                    errs_k[k].append(_decode_expected_err(er_k[b].cpu().numpy(), rec, hw_cache))
    rmse2 = float(np.sqrt(np.concatenate(errs2).mean()))
    ce2   = float(loss2_sum / max(n_batches, 1))
    rmse1 = float(np.sqrt(np.concatenate(errs1).mean())) if stacked else None
    ce1   = float(loss1_sum / max(n_batches, 1)) if stacked else None
    if want_k:
        multik = {k: (float(np.sqrt(np.concatenate(errs_k[k]).mean())),
                      float(lossk_sum[k] / max(n_batches, 1))) for k in want_k}
        return rmse2, ce2, rmse1, ce1, multik
    return rmse2, ce2, rmse1, ce1


### 9 · `_eval_model_pool` — the faithful seed-ensemble scorer

In [ ]:
# ============================================================================
# §C.1  Ensemble eval — seed-only (5-seed best_state average; the headline CV)
# ============================================================================
POSITION_BINS = 20

def _eval_model_pool(fold, ckpt_paths, k=1):
    """Average decoded expected_row over an arbitrary pool of checkpoints, each iterated to
    self-refeed depth `k` (deployment-matched HANDOFF_TRAIN_T / REFEED_TRAIN_T). k=1 -> a single
    forward pass (bit-identical to the original swap scorer).
    Returns (rmse, pos_rmse, well_residuals, errs_sq)."""
    models = []
    for p in ckpt_paths:
        state = torch.load(p, map_location=DEVICE)
        m = build_model(encoder_name=ENCODER, in_ch=IN_CH).to(DEVICE)
        m.load_state_dict(state); m.eval(); models.append(m)
    val_ds = RealValDataset(fold_splits[fold]['val'])
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4,
                            worker_init_fn=seed_worker)
    all_errs_sq = []
    pos_sum_sq = np.zeros(POSITION_BINS, dtype=np.float64)
    pos_count  = np.zeros(POSITION_BINS, dtype=np.int64)
    well_residuals = {}
    with torch.no_grad():
        for batch in val_loader:
            wids = batch['well_id']; B = batch['image'].size(0)
            img = batch['image'].to(DEVICE)
            ers = []
            for m in models:
                # iterate to depth k with the deployment-matched refeed/handoff temps;
                # k==1 -> _eval_iter_logits(...)[1] == m(img) (identity).
                if k > 1:
                    _kl, _ = _eval_iter_logits(m, img, k)
                    logits = _kl[k]
                else:
                    logits = m(img)
                probs = F.softmax(logits, dim=1)
                er = (probs * row_idx_t[None, :, None]).sum(dim=1).cpu().numpy()
                ers.append(er)
            er_mean = np.stack(ers, axis=0).mean(axis=0)          # == decode(mean probs)
            for b in range(B):
                wid = wids[b]; rec = well_images_real[wid]
                pred_col = rec['tvt_lo'] + (er_mean[b] + 0.5) * BIN_SIZE
                hw = pd.read_csv(well_hw_path(wid), usecols=['TVT','TVT_input'])
                n_hw = len(hw); bin_edges = np.linspace(0, n_hw, W + 1).astype(int)
                pred_full = np.zeros(n_hw, dtype=np.float32)
                for c in range(W): pred_full[bin_edges[c]:bin_edges[c+1]] = pred_col[c]
                post = hw['TVT_input'].isna().values
                true_post = hw['TVT'].values[post]; pred_post = pred_full[post]
                ok = ~np.isnan(true_post); resid = true_post[ok] - pred_post[ok]
                all_errs_sq.append(resid**2)
                if len(resid) > 0:
                    pos_norm = np.linspace(0, 1, len(resid), endpoint=False) + 0.5/len(resid)
                    bin_idx = np.clip((pos_norm * POSITION_BINS).astype(int), 0, POSITION_BINS-1)
                    for kk in range(POSITION_BINS):
                        sel = bin_idx == kk
                        if sel.any():
                            pos_sum_sq[kk] += float(np.sum(resid[sel]**2)); pos_count[kk] += int(sel.sum())
                well_residuals[wid] = resid
    all_errs_sq_concat = np.concatenate(all_errs_sq)
    rmse = float(np.sqrt(all_errs_sq_concat.mean()))
    pos_rmse = np.sqrt(pos_sum_sq / np.maximum(pos_count, 1))
    return rmse, pos_rmse, well_residuals, all_errs_sq_concat


### 10 · OOF config — three backbones (B7 / V2-L / V2-XL), no dilation


In [ ]:
# === OOF / ENSEMBLE CONFIG — three backbones (B7, V2-L, V2-XL) ================================
# SAME stacked 2-stage architecture, SAME decoder (DECODER_CHANNELS), NO dilation, NO pixelshuffle
# (interp='nearest'). The ONLY thing that differs between members is the ENCODER backbone. Each dir
# holds that member's per-(fold,seed) FINETUNE best_state checkpoints (files ckpt_f{fold}_s{seed}*.pt);
# ALL folds and ALL seeds present are auto-discovered. FILL IN the three dirs below. Leave a dir '' to
# SKIP that member (you can run 1, 2, or 3 — the blend cell adapts to whichever ran).
MODELS = {
    'V2-L k=3 T1.5': {'dir': '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt',  'encoder': 'tu-tf_efficientnetv2_l.in21k_ft_in1k'},
    'V2-L k=3 T1.5 Small Image': {'dir': '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt_H224_W448_bin1.1428571428571428',  'encoder': 'tu-tf_efficientnetv2_l.in21k_ft_in1k'},
    'V2-XL k=3 T1.5': {'dir': '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_effnetv2_xl_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt',  'encoder': 'tu-tf_efficientnetv2_xl.in21k_ft_in1k'},
    'L2 k=1 T1.5': {'dir': '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_l2_stack2_ema_25000pt_mdwarp42_bedwarp42_handoffT15_expa_pt',  'encoder': 'tu-tf_efficientnet_l2.ns_jft_in1k_475'},
}
OOF_INTERP = 'nearest'   # all members: native SMP decoder, no subpixel module (set once)
# NOTE: 'encoder' must match the string each member was TRAINED with, or strict checkpoint load fails.

_active, _blank = {}, []
for _name, _cfg in MODELS.items():
    _d = _cfg['dir'].strip()
    if not _d:
        _blank.append(_name); continue
    if not os.path.isdir(_d):
        raise FileNotFoundError(f'{_name}: not a directory: {_d}')
    _active[_name] = {'dir': _d, 'encoder': _cfg['encoder']}
if _active:
    print('Active members (dir set):')
    for _name, _cfg in _active.items():
        print(f'  {_name:6s} encoder={_cfg["encoder"]}')
        print(f'         dir={_cfg["dir"]}')
else:
    print('No dirs set yet — fill in MODELS[...]["dir"] for at least one member, then re-run.')
if _blank:
    print('Skipped (blank dir):', ', '.join(_blank))


# === INFERENCE CONTRACT + CACHE config ======================================================
# Depth and temperatures are PINNED (no sweep). Every member — cached or freshly inferred — must be
# k=3 self-refeed at T=1.0. The temps live in the model cell; assert them here so a stale edit can't
# silently change what the cache and the fresh path mean.
OOF_K = 3
assert float(HANDOFF_TRAIN_T) == 1.0 and float(REFEED_TRAIN_T) == 1.0, (
    f'this notebook is pinned to T=1.0; got handoff_T={HANDOFF_TRAIN_T} refeed_T={REFEED_TRAIN_T}')

# Cache-first: reuse oof_f{fold}_s{seed}.npz written by the training notebook's §C.2 cell.
USE_OOF_CACHE      = True    # False -> always re-run inference (checkpoints required)
CACHE_STRICT       = False    # reject cached files whose checkpoint_k/refeed_T/handoff_T disagree
SAVE_OOF_CACHE     = True    # after a fresh inference pass, write the per-seed npz for next time
CACHE_REQUIRE_SEED_MATCH = True   # cached seed set must equal the dir's ckpt_f*_s*.pt seed set
                                  # (set False if a dir ships OOF exports WITHOUT the checkpoints)
print(f'\ninference contract: k={OOF_K}  handoff_T={HANDOFF_TRAIN_T}  refeed_T={REFEED_TRAIN_T}   '
      f'| cache: use={USE_OOF_CACHE} strict={CACHE_STRICT} save={SAVE_OOF_CACHE}')


### 11 · Run OOF for every member


In [ ]:
# === OOF DRIVER — CACHE-FIRST, pinned to k=3 / T=1.0 ==========================================
# Order of preference per member:
#   1. cached per-seed exports  oof_f{fold}_s{seed}.npz  (written by the training notebook's §C.2 cell)
#   2. fresh inference via _eval_model_pool(fold, ckpts, k=OOF_K)   [+ optional cache write]
#
# WHY THE CACHE IS EXACT (not an approximation):
#   _eval_model_pool averages the per-seed expected rows `er`, then decodes
#       pred_col = tvt_lo + (er_mean + 0.5)*BIN_SIZE  ->  broadcast to MD rows  ->  resid = truth - pred.
#   Every step after `er` is AFFINE in `er` and uses per-well constants shared by all seeds
#   (tvt_lo, BIN_SIZE, bin_edges, the post-PS mask). Therefore
#       resid(mean_s er_s) == mean_s resid(er_s),
#   i.e. averaging the cached per-seed RESIDUALS reproduces the seed-ensemble residual bit-for-bit
#   (float assoc. aside). The cached `resid_post` is exactly the per-seed term of that mean.
import glob as _glob

def _dir_ckpts(d, fold):
    hits = sorted(_glob.glob(os.path.join(d, f'ckpt_f{fold}_s*.pt')))   # finetune best_state (excludes pretrain_*)
    if not hits:
        raise FileNotFoundError(f'no ckpt_f{fold}_s*.pt in {d}; dir has {sorted(os.listdir(d))[:10]}')
    return hits

def _seed_of(path, prefix):
    """Extract the seed token from ckpt_f{f}_s{seed}.pt / oof_f{f}_s{seed}.npz."""
    base = os.path.basename(path)
    return base.split('_s')[-1].split('.')[0]

def _dir_cached_oof(d, fold):
    return sorted(_glob.glob(os.path.join(d, f'oof_f{fold}_s*.npz')))

def _load_cached_fold(d, fold):
    """Load every cached per-seed OOF for `fold` and rebuild the seed-ensemble residuals.
    Returns (resid_by_well, rmse, n_seeds) or None if unusable (missing / provenance mismatch /
    seed set disagrees with the checkpoints)."""
    paths = _dir_cached_oof(d, fold)
    if not paths:
        return None
    if CACHE_REQUIRE_SEED_MATCH:
        try:
            ck_seeds = {_seed_of(p, 'ckpt') for p in _dir_ckpts(d, fold)}
            oof_seeds = {_seed_of(p, 'oof') for p in paths}
            if ck_seeds != oof_seeds:
                print(f'    fold {fold}: cache seed set {sorted(oof_seeds)} != ckpt seed set '
                      f'{sorted(ck_seeds)} -> recompute')
                return None
        except FileNotFoundError:
            pass          # no checkpoints in this dir: cache is all we have, accept it
    acc = {}              # wid -> list of per-seed residual arrays
    for p in paths:
        z = np.load(p, allow_pickle=True)
        if CACHE_STRICT:
            k_c = int(z['checkpoint_k']) if 'checkpoint_k' in z.files else -1
            rT  = float(z['refeed_T'])   if 'refeed_T'    in z.files else float('nan')
            hT  = float(z['handoff_T'])  if 'handoff_T'   in z.files else float('nan')
            if not (k_c == int(OOF_K) and rT == float(REFEED_TRAIN_T) and hT == float(HANDOFF_TRAIN_T)):
                print(f'    {os.path.basename(p)}: provenance k={k_c} refeed_T={rT} handoff_T={hT} '
                      f'!= (k={OOF_K}, {REFEED_TRAIN_T}, {HANDOFF_TRAIN_T}) -> recompute fold {fold}')
                return None
        wids = [str(w) for w in z['well_ids']]
        rp   = z['resid_post']
        for i, w in enumerate(wids):
            acc.setdefault(w, []).append(np.asarray(rp[i], dtype=np.float64))
    n_seeds = len(paths)
    resid_by_well = {}
    for w, lst in acc.items():
        if len(lst) != n_seeds:
            print(f'    fold {fold}: well {w} present in {len(lst)}/{n_seeds} cached seeds -> recompute')
            return None
        L = len(lst[0])
        if any(len(a) != L for a in lst):
            print(f'    fold {fold}: well {w} row-count mismatch across seeds -> recompute')
            return None
        resid_by_well[w] = np.mean(np.stack(lst, axis=0), axis=0)     # == seed-ensemble residual
    flat = np.concatenate([resid_by_well[w] for w in sorted(resid_by_well)])
    return resid_by_well, float(np.sqrt(np.mean(flat ** 2))), n_seeds

def _save_cached_fold(d, fold, ckpt_paths):
    """After a fresh pass, write per-seed npz so the next run is cache-hit. Uses the SAME single-model
    decode as the training notebook's §C.2 export (identical keys/format)."""
    for cp in ckpt_paths:
        seed = _seed_of(cp, 'ckpt')
        out_p = os.path.join(d, f'oof_f{fold}_s{seed}.npz')
        if os.path.exists(out_p):
            continue
        state = torch.load(cp, map_location=DEVICE)
        m = build_model(encoder_name=ENCODER, in_ch=IN_CH).to(DEVICE)
        m.load_state_dict(state); m.eval()
        loader = DataLoader(RealValDataset(fold_splits[fold]['val']), batch_size=8, shuffle=False,
                            num_workers=4, worker_init_fn=seed_worker)
        wl, er_l, pc_l, pp_l, rp_l, sq = [], [], [], [], [], []
        with torch.no_grad():
            for batch in loader:
                img = batch['image'].to(DEVICE)
                logits = _eval_iter_logits(m, img, OOF_K)[0][OOF_K] if OOF_K > 1 else m(img)
                er = (F.softmax(logits, dim=1) * row_idx_t[None, :, None]).sum(dim=1).cpu().numpy()
                for b in range(img.size(0)):
                    wid = batch['well_id'][b]; rec = well_images_real[wid]
                    pred_col = rec['tvt_lo'] + (er[b] + 0.5) * BIN_SIZE
                    hw = pd.read_csv(well_hw_path(wid), usecols=['TVT', 'TVT_input'])
                    n_hw = len(hw); be = np.linspace(0, n_hw, W + 1).astype(int)
                    pf = np.zeros(n_hw, dtype=np.float32)
                    for c in range(W): pf[be[c]:be[c+1]] = pred_col[c]
                    post = hw['TVT_input'].isna().values
                    tp = hw['TVT'].values[post]; pp = pf[post]
                    ok = ~np.isnan(tp); rs = (tp[ok] - pp[ok]).astype(np.float32)
                    wl.append(wid); er_l.append(er[b].astype(np.float32))
                    pc_l.append(pred_col.astype(np.float32)); pp_l.append(pp.astype(np.float32))
                    rp_l.append(rs); sq.append(rs.astype(np.float64) ** 2)
        def _obj(lst):
            a = np.empty(len(lst), dtype=object)
            for i, v in enumerate(lst): a[i] = v
            return a
        np.savez(out_p, well_ids=np.array(wl), er_row=_obj(er_l), pred_col=_obj(pc_l),
                 pred_post=_obj(pp_l), resid_post=_obj(rp_l),
                 rmse=float(np.sqrt(np.concatenate(sq).mean())), fold=fold, seed=seed,
                 checkpoint_k=int(OOF_K), refeed_T=float(REFEED_TRAIN_T),
                 handoff_T=float(HANDOFF_TRAIN_T))
        del m
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        print(f'    cached -> {os.path.basename(out_p)}')

def run_oof(model_dir, encoder, interp_mode='nearest'):
    """Per-fold seed-ensemble OOF at the pinned k=OOF_K / T=1.0, cache-first."""
    global DECODER_INTERP, ENCODER
    DECODER_INTERP = interp_mode; ENCODER = encoder
    resid_by_well = {}; per_fold = {}; errs_all = []; n_cached = 0
    print(f'[{os.path.basename(model_dir.rstrip("/"))}]  encoder={encoder}  interp={interp_mode}  k={OOF_K}')
    for f in ALL_FOLDS:
        hit = _load_cached_fold(model_dir, f) if USE_OOF_CACHE else None
        if hit is not None:
            wres, rmse, n_seeds = hit
            # NOTE: _eval_model_pool's 4th return is SQUARED errors (full-CV pools them as
            # sqrt(mean(sq))), so the cache path must square here to stay in the same space.
            errs = np.concatenate([wres[w] for w in sorted(wres)]) ** 2
            src_tag = f'CACHE ({n_seeds} seeds)'
            n_cached += 1
        else:
            cps = _dir_ckpts(model_dir, f)
            rmse, pos_rmse, wres, errs = _eval_model_pool(f, cps, k=OOF_K)
            src_tag = f'infer ({len(cps)} seeds)'
            if SAVE_OOF_CACHE:
                _save_cached_fold(model_dir, f, cps)
        per_fold[f] = rmse; resid_by_well.update(wres); errs_all.append(errs)
        print(f'  fold {f}: ens RMSE = {rmse:.4f}   [{src_tag} x {len(fold_splits[f]["val"])} val wells]')
    full = float(np.sqrt(np.concatenate(errs_all).mean()))
    print(f'  -> FULL-CV OOF RMSE (k={OOF_K}) = {full:.4f}   [{n_cached}/{len(ALL_FOLDS)} folds from cache]\n')
    return resid_by_well, per_fold, full


oof_results = {}    # name -> (resid_by_well, per_fold, full)
print(f'>>> OOF driver: k={OOF_K}, handoff_T={HANDOFF_TRAIN_T}, refeed_T={REFEED_TRAIN_T}\n')
for _name, _cfg in _active.items():
    print(f'===== {_name} =====')
    oof_results[_name] = run_oof(_cfg['dir'], _cfg['encoder'], OOF_INTERP)
print('OOF complete for:', ', '.join(oof_results) or '(none — set dirs first)')


In [ ]:
import numpy as np, os
RH_DIR = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt'
z = np.load(os.path.join(RH_DIR, 'result_f0_s102.npz'))
print('training best_rmse (sel@k3,T1.5):', float(z['best_rmse']))

### 12 · Correlation & blend

In [ ]:
# === CORRELATION + BLEND — N members (residual space == prediction space; shared GT cancels) ===
# Scales to any member count: aliases M1..Mn keep tables readable, and the optimal blend is
# solved EXACTLY for any N via an active-set QP on the residual Gram matrix (blend SSE = w^T G w,
# convex on the simplex; every face's stationary point is enumerated — no weight grid).
import itertools as _it

names = list(oof_results)
assert names, 'no OOF results — set dirs in the config cell and run the OOF driver first.'
res = {n: oof_results[n][0] for n in names}                 # name -> {wid: residual array}
wids = sorted(set.intersection(*[set(res[n]) for n in names]))
for n in names:
    assert all(len(res[n][w]) == len(res[names[0]][w]) for w in wids), f'{n}: per-well length mismatch'
E  = {n: np.concatenate([res[n][w] for w in wids]) for n in names}      # name -> flat residual vector
folds_of = {w: f for f in ALL_FOLDS for w in fold_splits[f]['val']}
fw = np.concatenate([np.full(len(res[names[0]][w]), folds_of[w]) for w in wids])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))

M = len(names)
alias = {n: f'M{i+1}' for i, n in enumerate(names)}
print('=== members (aliases used in every table below) ===')
for n in names:
    print(f'  {alias[n]:>3s} = {n:<28s} full-CV RMSE {rmse(E[n]):.4f}')
best_single = min(names, key=lambda n: rmse(E[n]))
print(f'  best single: {alias[best_single]} ({rmse(E[best_single]):.4f})')

def opt_simplex_qp(G):
    """Exact min of w^T G w on the simplex (w>=0, sum w=1), any dimension.
    Active-set enumeration: on each support subset S the stationary point solves
    G_S w = const*1; keep it if feasible; the global optimum is among these
    (G is PSD -> convex QP; vertices are the |S|=1 cases). Returns (w, sse)."""
    m = G.shape[0]
    best_w, best_v = None, np.inf
    idx = np.arange(m)
    for r in range(1, m + 1):
        for S in _it.combinations(idx, r):
            GS = G[np.ix_(S, S)]
            try:
                x = np.linalg.lstsq(GS, np.ones(r), rcond=None)[0]
            except np.linalg.LinAlgError:
                continue
            s = x.sum()
            if not np.isfinite(s) or abs(s) < 1e-12:
                continue
            x = x / s
            if (x < -1e-9).any():
                continue
            x = np.clip(x, 0.0, None); x = x / x.sum()
            w = np.zeros(m); w[list(S)] = x
            v = float(w @ G @ w)
            if v < best_v: best_v, best_w = v, w
    return best_w, best_v

if M >= 2:
    # --- diversity as a compact matrix: all-rows in the LOWER triangle, worst-decile UPPER ---
    print('\n=== error-correlation matrix (lower = all rows | upper = shared worst-decile rows) ===')
    C_all = np.eye(M); C_wd = np.eye(M)
    for i, j in _it.combinations(range(M), 2):
        a, b = names[i], names[j]
        C_all[i, j] = C_all[j, i] = np.corrcoef(E[a], E[b])[0, 1]
        comb = np.abs(E[a]) + np.abs(E[b]); wd = comb >= np.quantile(comb, 0.90)
        C_wd[i, j] = C_wd[j, i] = np.corrcoef(E[a][wd], E[b][wd])[0, 1]
    hdr = '        ' + ' '.join(f'{alias[n]:>6s}' for n in names)
    print(hdr)
    for i, n in enumerate(names):
        cells = []
        for j in range(M):
            if j < i:   cells.append(f'{C_all[i, j]:6.3f}')     # lower: all rows
            elif j > i: cells.append(f'{C_wd[i, j]:6.3f}')      # upper: worst decile
            else:       cells.append(f'{"·":>6s}')
        print(f'  {alias[n]:>4s}  ' + ' '.join(cells))
    print('  (low upper-triangle values = tail diversity, the kind blending can spend)')

    # --- Gram matrix once; everything downstream is exact and instant ---
    Emat = np.stack([E[n] for n in names], axis=1)              # rows x M
    N_rows = Emat.shape[0]
    G_all = Emat.T @ Emat

    eqw = np.full(M, 1.0 / M)
    print(f'\n=== equal-weight blend ===\n  1/{M} each -> {np.sqrt(eqw @ G_all @ eqw / N_rows):.4f} ft'
          f'   (best single {rmse(E[best_single]):.4f})')

    w_opt, v_opt = opt_simplex_qp(G_all)
    print('\n=== optimal full-CV blend (exact simplex QP, any N) ===')
    print('  weights: ' + '  '.join(f'{alias[n]}={w_opt[i]:.3f}' for i, n in enumerate(names)
                                    if w_opt[i] > 5e-4) +
          ('' if all(w_opt > 5e-4) else '   (others 0)'))
    r_opt = float(np.sqrt(v_opt / N_rows))
    print(f'  blend RMSE = {r_opt:.4f} ft   | best single {rmse(E[best_single]):.4f}'
          f'   | gain vs best single = {rmse(E[best_single]) - r_opt:+.4f} ft')

    print('\n=== per-fold OOF RMSE + per-fold optimal blend ===')
    print('  fold  ' + ' '.join(f'{alias[n]:>8s}' for n in names) + '     blend    weights (per-fold opt)')
    for f in ALL_FOLDS:
        m_ = fw == f
        Gf = Emat[m_].T @ Emat[m_]
        wf, vf = opt_simplex_qp(Gf)
        cells = ' '.join(f'{rmse(E[n][m_]):8.4f}' for n in names)
        wstr  = ','.join(f'{x:.2f}' for x in wf)
        print(f'   {f}   {cells}   {np.sqrt(vf / m_.sum()):8.4f}   [{wstr}]')

    print('\nDecision: a blend below the BEST single — especially with low worst-decile (upper-triangle)')
    print('correlation — means the extra backbone(s) earn an ensemble slot. A blend pinned at the best')
    print('single => errors correlated, pooling buys nothing. (Uncorrelated backbones are the premise.)')
else:
    print('\nOnly one member active — set more dirs to get correlation + blend.')


In [ ]:
# ============================================================================
# ENSEMBLE DECOMPOSITION — datum/drift split + toe (position) profile
# Paste AFTER the OOF driver (needs oof_results). Self-contained: recomputes the
# optimal blend internally, then shows WHERE ensembling actually helps.
# ============================================================================
import numpy as np

names = list(oof_results)
assert names, 'run the OOF driver first (oof_results is empty).'
res  = {n: oof_results[n][0] for n in names}          # name -> {wid: per-MD-row post-PS residual}
wids = sorted(set.intersection(*[set(res[n]) for n in names]))
for n in names:
    assert all(len(res[n][w]) == len(res[names[0]][w]) for w in wids), f'{n}: per-well length mismatch'
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))

# ---- per-well split: r_w = datum_w (well mean = offset) + drift_w (de-meaned = shape) ----
# Exact orthogonal decomposition: total^2 = datum^2 + drift^2 (drift is per-well zero-mean).
def datum_drift(rbw):
    dat_sq = dft_sq = 0.0; ntot = 0
    for w in wids:
        r = rbw[w]; n = len(r)
        if n == 0: continue
        d = float(r.mean())
        dat_sq += n * d * d
        dft_sq += float(np.sum((r - d) ** 2))
        ntot   += n
    return (np.sqrt(dat_sq / ntot), np.sqrt(dft_sq / ntot), np.sqrt((dat_sq + dft_sq) / ntot))

# ---- toe profile: bin each well's post-PS rows by normalized MD position (0=heel ... 1=toe) ----
TOE_EDGES = [0.0, 0.25, 0.50, 0.75, 0.90, 1.0]        # last band = toe 90-100%
_NB = len(TOE_EDGES) - 1
def toe_profile(rbw):
    sq = np.zeros(_NB); cnt = np.zeros(_NB, dtype=np.int64)
    for w in wids:
        r = rbw[w]; n = len(r)
        if n == 0: continue
        pos = (np.arange(n) + 0.5) / n
        idx = np.clip(np.searchsorted(TOE_EDGES, pos, side='right') - 1, 0, _NB - 1)
        for b in range(_NB):
            m = idx == b
            if m.any(): sq[b] += float(np.sum(r[m] ** 2)); cnt[b] += int(m.sum())
    return np.sqrt(sq / np.maximum(cnt, 1)), cnt

# ---- blends: equal-weight + optimal full-CV blend (EXACT simplex QP, any member count) ----
E = {n: np.concatenate([res[n][w] for w in wids]) for n in names}
_alias27 = {n: f'M{i+1}' for i, n in enumerate(names)}
_Emat27 = np.stack([E[n] for n in names], axis=1)
_G27 = _Emat27.T @ _Emat27
if 'opt_simplex_qp' not in globals():
    import itertools as _it27
    def opt_simplex_qp(G):
        m = G.shape[0]; best_w, best_v = None, np.inf
        for r in range(1, m + 1):
            for S in _it27.combinations(range(m), r):
                GS = G[np.ix_(S, S)]
                try: x = np.linalg.lstsq(GS, np.ones(r), rcond=None)[0]
                except np.linalg.LinAlgError: continue
                s = x.sum()
                if not np.isfinite(s) or abs(s) < 1e-12: continue
                x = x / s
                if (x < -1e-9).any(): continue
                x = np.clip(x, 0.0, None); x = x / x.sum()
                w = np.zeros(m); w[list(S)] = x
                v = float(w @ G @ w)
                if v < best_v: best_v, best_w = v, w
        return best_w, best_v
eq = {n: 1.0 / len(names) for n in names}
_wv27, _ = opt_simplex_qp(_G27)
wopt = {n: float(_wv27[i]) for i, n in enumerate(names)}
blend_well = lambda weights: {w: sum(weights[n] * res[n][w] for n in names) for w in wids}

rows = [(n, res[n]) for n in names]
rows += [('blend(eq)', blend_well(eq)), ('blend(opt)', blend_well(wopt))]

# ---- 1) DATUM vs DRIFT ----
print('=== DATUM vs DRIFT  (per-well offset vs within-well shape; total^2 = datum^2 + drift^2) ===')
print(f'  {"member":12s}  {"total":>7s}  {"datum":>7s}  {"drift":>7s}')
for label, rbw in rows:
    dat, dft, tot = datum_drift(rbw)
    print(f'  {_alias27.get(label, label):12s}  {tot:7.4f}  {dat:7.4f}  {dft:7.4f}')
bs = min(names, key=lambda n: datum_drift(res[n])[2])
bdat, bdft, btot = datum_drift(res[bs])
odat, odft, otot = datum_drift(blend_well(wopt))
print(f'  best single = {_alias27[bs]}   |   opt weights: ' + ', '.join(f'{_alias27[n]}={wopt[n]:.2f}' for n in names))
print(f'  GAIN (best single -> opt blend):  total {btot-otot:+.4f}  |  '
      f'datum {bdat-odat:+.4f}  |  drift {bdft-odft:+.4f}')
print('  datum = absolute-position/offset lane (the wall); drift = within-well shape lane (the open one).')

# ---- 2) TOE PROFILE ----
print('\n=== TOE PROFILE  (RMSE by normalized post-PS position; 0=heel ... 1=toe) ===')
labels = [f'{int(TOE_EDGES[i]*100)}-{int(TOE_EDGES[i+1]*100)}%' for i in range(_NB)]
_, cnt0 = toe_profile(rows[0][1])
print('  band        ' + '  '.join(f'{l:>9s}' for l in labels))
print('  rows        ' + '  '.join(f'{c:9,d}' for c in cnt0))
for label, rbw in rows:
    pr, _ = toe_profile(rbw)
    print(f'  {_alias27.get(label, label):10s}  ' + '  '.join(f'{v:9.4f}' for v in pr))
pr_bs, _  = toe_profile(res[bs]); pr_opt, _ = toe_profile(blend_well(wopt))
print(f'  toe-band (90-100%) gain, best single {bs} -> opt blend: {pr_bs[-1]-pr_opt[-1]:+.4f} ft')
print('  Tail-concentrated metric: the 90-100% toe band is where most squared error lives — watch it move.')


### 13 · Spatial / similarity CV-inflation diagnostic
Does per-well OOF error depend on distance (or typewell similarity) to the fold-train set — and do test wells sit at val-like train-distances? Verdict rules R1–R3 are pre-registered in the cell header.

In [ ]:
# ============================================================================
# SPATIAL / SIMILARITY CV-INFLATION DIAGNOSTIC
# Paste AFTER the OOF driver (needs oof_results, fold_splits, ALL_FOLDS, well paths).
#
# QUESTION: is the OOF CV optimistic because validation wells sit near (or are
# geologically similar to) training wells? The model never receives X/Y — but it
# doesn't need to: nearby wells share the formation surface, GR character, and
# typewell references, so a val well with a near-twin in train is an easier
# (interpolation) problem than a geographically isolated test well (extrapolation).
# Two measurements answer this:
#   (A) does per-well error INCREASE with distance/dissimilarity to that well's
#       own fold-train set?  (if flat -> proximity isn't buying the model anything)
#   (B) do TEST wells sit at train-distances like val wells do?  (if yes, even a
#       positive (A) doesn't bias CV; if test is systematically FARTHER, the
#       far-subset / distance-matched CV numbers below are the better estimate)
#
# PRE-REGISTERED verdict rules (fixed before looking at the numbers):
#   R1  proximity signal:  Spearman rho(d, per-well RMSE) >= +0.15 with perm-p < 0.01
#   R2  material inflation: pooled RMSE(far half) - pooled RMSE(near half) >= +0.30 ft
#       with a bootstrap 95% CI excluding 0   (split at median nearest-train distance)
#   R3  shift risk: median test nearest-train distance > p75 of val nearest-train distance
#   CV is flagged INFLATED only if (R1 or R2) AND R3 — a proximity gradient alone is
#   harmless when test wells are just as close to train as val wells are.
# ============================================================================
import hashlib
import numpy as np, pandas as pd

RNG_DIAG   = np.random.RandomState(1234)
N_PERM     = 10_000          # permutation draws for Spearman p
N_BOOT     = 5_000           # bootstrap draws for the far-near CI
TRAJ_SUB   = 25              # trajectory subsample stride (rows) for min-distance
TRAJ_PRE_K = 30              # exact trajectory distance vs the K centroid-nearest train wells
TWSIG_L    = 256             # typewell GR signature resample length

# ---- 1 · error vector: equal-weight blend of whatever members ran (falls back to single)
_names = list(oof_results)
assert _names, 'no OOF results — run the OOF driver first.'
_res   = {n: oof_results[n][0] for n in _names}
_wids  = sorted(set.intersection(*[set(_res[n]) for n in _names]))
for _n in _names:
    assert all(len(_res[_n][w]) == len(_res[_names[0]][w]) for w in _wids), f'{_n}: length mismatch'
_blend_res = {w: np.mean(np.stack([_res[n][w] for n in _names]), axis=0) for w in _wids}
_well_rmse = {w: float(np.sqrt(np.mean(_blend_res[w] ** 2))) for w in _wids}
_folds_of  = {w: f for f in ALL_FOLDS for w in fold_splits[f]['val']}
_wids      = [w for w in _wids if w in _folds_of]          # only wells with a known fold
_pooled    = float(np.sqrt(np.mean(np.concatenate([_blend_res[w] for w in _wids]) ** 2)))
print(f'diagnostic error vector: equal-weight blend of {_names} | {len(_wids)} val wells '
      f'| pooled post-PS RMSE = {_pooled:.4f} ft')

# ---- 2 · per-well geometry + typewell signature caches (train + val [+ test if present])
def _load_xy(wid, hw_dir):
    p = os.path.join(hw_dir, wid + HW_SUFFIX)
    df = pd.read_csv(p, usecols=['X', 'Y']).dropna()
    assert len(df) > 0, f'{wid}: no finite X/Y rows'
    pts = df.values[::TRAJ_SUB].astype(np.float64)
    return {'cent': df.values.mean(axis=0), 'pts': pts}

def _load_twsig(wid, hw_dir):
    p = os.path.join(hw_dir, wid + TW_SUFFIX)
    with open(p, 'rb') as fh:
        digest = hashlib.md5(fh.read()).hexdigest()
    tw = pd.read_csv(p).dropna(subset=['TVT', 'GR']).sort_values('TVT')
    g  = np.interp(np.linspace(tw['TVT'].min(), tw['TVT'].max(), TWSIG_L),
                   tw['TVT'].values, tw['GR'].values)
    g  = (g - g.mean()) / (g.std() + 1e-9)
    return {'hash': digest, 'sig': g.astype(np.float64)}

print('caching X/Y + typewell signatures for all train wells ...')
_geo, _tws = {}, {}
_all_train_wells = sorted({w for f in ALL_FOLDS for s in ('train', 'val') for w in fold_splits[f][s]})
for _w in tqdm(_all_train_wells, desc='geometry'):
    _geo[_w] = _load_xy(_w, TRAIN_DIR); _tws[_w] = _load_twsig(_w, TRAIN_DIR)

# ---- 3 · per-val-well similarity features vs its OWN fold-train set
def _cos_dist(a, b):  # signatures are z-scored, so this is 1 - Pearson-like alignment
    return float(1.0 - np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12))

_rows = []
for _w in tqdm(_wids, desc='val-vs-train distances'):
    _tr = fold_splits[_folds_of[_w]]['train']
    _c  = _geo[_w]['cent']
    _dc = np.array([np.hypot(*(np.asarray(_geo[t]['cent']) - _c)) for t in _tr])
    _near_order = np.argsort(_dc)
    # exact min trajectory-to-trajectory distance over the K centroid-nearest candidates
    _dt = np.inf
    for _j in _near_order[:TRAJ_PRE_K]:
        _tp = _geo[_tr[_j]]['pts']; _vp = _geo[_w]['pts']
        _dt = min(_dt, float(np.sqrt(
            ((_vp[:, None, :] - _tp[None, :, :]) ** 2).sum(-1)).min()))
    _dsig = min(_cos_dist(_tws[_w]['sig'], _tws[t]['sig']) for t in _tr)
    _dup  = any(_tws[_w]['hash'] == _tws[t]['hash'] for t in _tr)
    _rows.append({'wid': _w, 'fold': _folds_of[_w], 'rmse': _well_rmse[_w],
                  'n_post': len(_blend_res[_w]),
                  'd_cent': float(_dc.min()), 'd_traj': _dt,
                  'd_twsig': _dsig, 'tw_dup': bool(_dup)})
D = pd.DataFrame(_rows).set_index('wid')
print(f"\nnearest-train distance (val): centroid ft  p25/p50/p75 = "
      f"{D['d_cent'].quantile(.25):,.0f} / {D['d_cent'].quantile(.50):,.0f} / "
      f"{D['d_cent'].quantile(.75):,.0f}   | trajectory-min p50 = {D['d_traj'].median():,.0f}")
if D['tw_dup'].any():
    print(f"typewell EXACT duplicates with a train well: {int(D['tw_dup'].sum())}/{len(D)} val wells")

# ---- 4 · statistics -------------------------------------------------------------------
def _spearman_perm(x, y, n_perm=N_PERM):
    rx, ry = pd.Series(x).rank().values, pd.Series(y).rank().values
    rho = float(np.corrcoef(rx, ry)[0, 1])
    null = np.empty(n_perm)
    for i in range(n_perm):
        null[i] = np.corrcoef(rx, RNG_DIAG.permutation(ry))[0, 1]
    p = float((np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1))
    return rho, p

print('\n=== (A) per-well RMSE vs distance/dissimilarity to own fold-train set ===')
print('  key       Spearman rho   perm-p      (positive rho = farther/less-similar -> worse)')
_rhos = {}
for _k in ('d_cent', 'd_traj', 'd_twsig'):
    _rho, _p = _spearman_perm(D[_k].values, D['rmse'].values)
    _rhos[_k] = (_rho, _p)
    print(f'  {_k:8s}   {_rho:+8.3f}     {_p:8.4f}')

# partial control: rank-regress well "size/shape" proxies out of both sides, re-correlate.
# (a raw rho can reflect difficulty clustering, not proximity benefit — see verdict note)
_ctl = np.stack([D['n_post'].rank().values,
                 D['d_twsig'].rank().values], axis=1).astype(float)
def _partial_rho(xk):
    rx = pd.Series(D[xk]).rank().values.astype(float)
    ry = D['rmse'].rank().values.astype(float)
    A  = np.column_stack([np.ones(len(_ctl)), _ctl])
    rx = rx - A @ np.linalg.lstsq(A, rx, rcond=None)[0]
    ry = ry - A @ np.linalg.lstsq(A, ry, rcond=None)[0]
    return float(np.corrcoef(rx, ry)[0, 1])
print(f"  d_cent partial rho (controls: n_post rows, tw-signature NN dist) = "
      f"{_partial_rho('d_cent'):+.3f}")

print('\n  d_cent quintile -> per-well RMSE (median [n])')
D['_q'] = pd.qcut(D['d_cent'], 5, labels=False, duplicates='drop')
for _q, _g in D.groupby('_q'):
    print(f'    Q{int(_q)+1}  d<= {_g["d_cent"].max():>9,.0f} ft   '
          f'median {_g["rmse"].median():.3f} ft   [n={len(_g)}]')

# near/far pooled contrast + bootstrap CI (rows pooled within halves, wells resampled)
_med = D['d_cent'].median()
_near_w = [w for w in D.index if D.at[w, 'd_cent'] <= _med]
_far_w  = [w for w in D.index if D.at[w, 'd_cent'] >  _med]
_pool   = lambda ws: float(np.sqrt(np.mean(np.concatenate([_blend_res[w] for w in ws]) ** 2)))
_near_r, _far_r = _pool(_near_w), _pool(_far_w)
_diffs = np.empty(N_BOOT)
for _i in range(N_BOOT):
    _bn = [RNG_DIAG.choice(_near_w) for _ in _near_w]
    _bf = [RNG_DIAG.choice(_far_w)  for _ in _far_w]
    _diffs[_i] = _pool(_bf) - _pool(_bn)
_lo, _hi = np.percentile(_diffs, [2.5, 97.5])
print(f'\n  pooled RMSE  NEAR half = {_near_r:.4f}   FAR half = {_far_r:.4f}   '
      f'diff = {_far_r - _near_r:+.4f} ft   (boot 95% CI [{_lo:+.3f}, {_hi:+.3f}])')
_far_q = D['d_cent'].quantile(0.75)
_farq_w = [w for w in D.index if D.at[w, 'd_cent'] > _far_q]
print(f'  far-QUARTILE pooled RMSE = {_pool(_farq_w):.4f}  (vs full CV {_pooled:.4f}) '
      f'-> proxy for a fully extrapolative test set')
if D['tw_dup'].any() and (~D['tw_dup']).any():
    print(f"  typewell-duplicate wells pooled RMSE = "
          f"{_pool([w for w in D.index if D.at[w,'tw_dup']]):.4f}  vs non-dup "
          f"{_pool([w for w in D.index if not D.at[w,'tw_dup']]):.4f}")

# ---- 5 · (B) where does TEST sit relative to train? -----------------------------------
_test_dir = os.path.join(DATA_ROOT, 'test')
R3 = None
if os.path.isdir(_test_dir):
    _test_wells = list_wells(_test_dir)
    print(f'\n=== (B) test-set position: {len(_test_wells)} test wells found ===')
    _tr_cents = np.stack([_geo[w]['cent'] for w in _all_train_wells])
    _dtest = []
    for _w in tqdm(_test_wells, desc='test distances'):
        _g = _load_xy(_w, _test_dir)
        _dtest.append(float(np.hypot(*(_tr_cents - _g['cent']).T).min()))
    _dtest = np.asarray(_dtest)
    print(f'  nearest-ANY-train distance (centroid ft):')
    print(f'    val  p25/p50/p75 = {D["d_cent"].quantile(.25):,.0f} / '
          f'{D["d_cent"].quantile(.50):,.0f} / {D["d_cent"].quantile(.75):,.0f}')
    print(f'    test p25/p50/p75 = {np.percentile(_dtest,25):,.0f} / '
          f'{np.percentile(_dtest,50):,.0f} / {np.percentile(_dtest,75):,.0f}')
    R3 = bool(np.median(_dtest) > D['d_cent'].quantile(0.75))
    # distance-matched CV: reweight val wells so their d_cent histogram matches test's
    _edges = np.quantile(np.concatenate([D['d_cent'].values, _dtest]),
                         np.linspace(0, 1, 9))
    _edges[0], _edges[-1] = -np.inf, np.inf
    _bv = np.digitize(D['d_cent'].values, _edges[1:-1])
    _bt = np.digitize(_dtest, _edges[1:-1])
    _wt = np.zeros(len(D))
    for _b in range(len(_edges) - 1):
        _mv, _pt = (_bv == _b), float(np.mean(_bt == _b))
        if _mv.any():
            _wt[_mv] = _pt / max(float(np.mean(_mv)), 1e-9)
    _sq = np.array([np.mean(_blend_res[w] ** 2) for w in D.index])
    _n  = np.array([len(_blend_res[w]) for w in D.index], dtype=float)
    _dm = float(np.sqrt(np.sum(_wt * _n * _sq) / np.sum(_wt * _n)))
    print(f'  DISTANCE-MATCHED CV (val reweighted to the test distance profile) = {_dm:.4f} ft'
          f'   (unweighted CV {_pooled:.4f})')
else:
    print(f'\n=== (B) no test dir at {_test_dir} — R3 (shift risk) not evaluable here. ===')

# ---- 6 · verdict ----------------------------------------------------------------------
R1 = any(r >= 0.15 and p < 0.01 for r, p in _rhos.values())
R2 = (_far_r - _near_r >= 0.30) and (_lo > 0)
print('\n=== VERDICT (pre-registered rules R1-R3 in the header) ===')
print(f'  R1 proximity signal (rho>=+0.15, p<0.01 on any key): {"FIRED" if R1 else "no"}')
print(f'  R2 material near/far gap (>=+0.30 ft, CI>0):          {"FIRED" if R2 else "no"}')
print(f'  R3 test farther than val (med(test) > p75(val)):      '
      f'{"FIRED" if R3 else ("no" if R3 is not None else "n/a — no test dir")}')
if (R1 or R2) and (R3 is True):
    print('  -> CV INFLATION FLAGGED: error grows with train-distance AND test is farther out.')
    print('     Use the far-quartile / distance-matched numbers above as the deployment estimate.')
elif (R1 or R2):
    print('  -> a proximity/difficulty gradient exists, but with no evidence test is farther')
    print('     than val, CV is not thereby biased. NOTE: rho here conflates proximity BENEFIT')
    print('     with spatial difficulty CLUSTERING (nearby wells share hard geology); the near/far')
    print('     pooled contrast (R2) is the deployment-relevant number, the rho is descriptive.')
else:
    print('  -> no distance dependence detected: per-well error is flat in train-proximity, so')
    print('     val wells sitting near train wells are not making CV optimistic at this scale.')


### 16 · PhaseJitter TTA — per-phase pass + composition search (V2-L, k=1)
One pass renders all 8 phase variants per well (batched — phases share each seed's forward), keeps per-phase residuals, then composes offline: singles, the deployed full-8 comb, prefix marginal-adds, rotated best subset, rotated free weights. Prices the CV tax at phase granularity; the §32.9 board benefit is invisible on OOF by construction and the read-out says so.

In [ ]:
# ============================================================================
# §17 · FP16 DAMAGE EVALUATION — paired fp32 vs autocast half precision (V2-L)
# Paste beneath the §16 PhaseJitter cell (needs cells 1-11: fold_splits,
# well_images_real, build_model, _dir_ckpts, decode constants).
#
# QUESTION: if the submission kernel runs members under autocast to fit the
# clock, what does half precision cost in accuracy — and buy in time/memory?
# DESIGN: for every fold/seed/batch, TWO forwards on the SAME images at the
# deployed depth (k=FP_K, handoff FP_T, refeed 1.0): once fp32, once autocast.
# Paired by construction — every difference is precision, nothing else.
# Single-pass (no phase sweep): TTA averages 8 decoded outputs in fp64 numpy,
# so per-forward noise only shrinks under the comb; single-pass is the
# conservative (worst-case) read.
#
# REPORTS: pooled RMSE fp32 vs fp16 + paired-bootstrap CI on the delta;
# prediction-level divergence |pred16 - pred32| (mean/p95/max ft, truth-free);
# per-well paired deltas incl. worst-decile wells; per-fold table; measured
# wall-time speedup + peak CUDA memory both ways.
#
# PRE-REGISTERED verdict (fixed before looking):
#   SAFE to deploy autocast if ALL of: |pooled delta| <= 0.010 ft;
#   per-row divergence p95 <= 0.05 ft; worst single-well RMSE delta <= 0.05 ft.
#   Otherwise FLAG — do not ship fp16 on this member without a mechanism look.
# NOTE: speedup here is THIS runtime's GPU; re-measure on Kaggle's GPU before
# betting the clock on it. The replication check: fp32 pooled should match the
# §16 ph0.0 number at the same k/T.
# ============================================================================
import time as _tm

FP_DIR    = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_effnetv2_l_stack2_ema_25000pt_mdwarp36_bedwarp36_unroll50_deepsup_handoffT15_expa_pt_H224_W448_bin1.1428571428571428'
FP_ENCODER = 'tu-tf_efficientnetv2_l.in21k_ft_in1k'
FP_K      = 3                       # deployed refeed depth
FP_T      = 1.5                   # handoff T (match the §16 probe for the replication check)
FP_REFEED = 1.0
FP_FOLDS  = list(ALL_FOLDS)         # [0] for a quick look
FP_DTYPE  = 'float16'               # 'float16' (CUDA) | 'bfloat16'
N_BOOT_FP = 2000
_fprng    = np.random.RandomState(1616)

assert '_dir_ckpts' in globals(), 'run the OOF driver cell first — needs _dir_ckpts'
assert os.path.isdir(FP_DIR), f'not a directory: {FP_DIR}'
ENCODER = FP_ENCODER; DECODER_INTERP = 'nearest'
_amp_dtype = torch.float16 if FP_DTYPE == 'float16' else torch.bfloat16
_amp_dev   = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
if _amp_dev == 'cpu' and _amp_dtype == torch.float16:
    print('NOTE: CPU autocast does not support float16 — falling back to bfloat16 for this run.')
    _amp_dtype = torch.bfloat16
print(f'fp16 damage eval: k={FP_K} T={FP_T}/{FP_REFEED}, folds={FP_FOLDS}, '
      f'autocast={_amp_dev}/{str(_amp_dtype).split(".")[-1]}\n  dir={FP_DIR}')

def _fp_forward(models, img):
    """Seed-mean expected-row at the deployed depth. Returns (W,)-per-image er (numpy)."""
    er_sum = None
    for _m in models:
        _l1 = _m.stage1(img).squeeze(1)
        _post = F.softmax(_l1 / float(FP_T), dim=1)
        _l2 = _m._stage2_from_post(img, _post)
        for _r in range(2, FP_K + 1):
            _post = F.softmax(_l2 / float(FP_REFEED), dim=1)
            _l2 = _m._stage2_from_post(img, _post)
        _er = (F.softmax(_l2.float(), dim=1) * row_idx_t[None, :, None]).sum(dim=1)
        er_sum = _er if er_sum is None else er_sum + _er
    return (er_sum / len(models)).cpu().numpy()

_res32, _res16, _pdiv = {}, {}, []       # wid -> residuals; per-row |pred16-pred32| samples
_t32 = _t16 = 0.0; _mem = {}
for _f in FP_FOLDS:
    _cps = _dir_ckpts(FP_DIR, _f)
    _models = []
    for _p in _cps:
        _m = build_model(encoder_name=FP_ENCODER, in_ch=IN_CH).to(DEVICE)
        _m.load_state_dict(torch.load(_p, map_location=DEVICE)); _m.eval(); _models.append(_m)
    _loader = DataLoader(RealValDataset(fold_splits[_f]['val']), batch_size=8, shuffle=False,
                         num_workers=4, worker_init_fn=seed_worker)
    with torch.no_grad():
        for _batch in tqdm(_loader, desc=f'fold {_f} ({len(_cps)} seeds, fp32+amp paired)'):
            _img = _batch['image'].to(DEVICE)
            if DEVICE.type == 'cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
            _t0 = _tm.time()
            _er32 = _fp_forward(_models, _img)
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            _t32 += _tm.time() - _t0
            if DEVICE.type == 'cuda': _mem['fp32'] = max(_mem.get('fp32', 0), torch.cuda.max_memory_allocated())
            if DEVICE.type == 'cuda': torch.cuda.reset_peak_memory_stats()
            _t0 = _tm.time()
            with torch.autocast(_amp_dev, dtype=_amp_dtype):
                _er16 = _fp_forward(_models, _img)
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            _t16 += _tm.time() - _t0
            if DEVICE.type == 'cuda': _mem['amp'] = max(_mem.get('amp', 0), torch.cuda.max_memory_allocated())
            for _b in range(_img.size(0)):
                _wid = _batch['well_id'][_b]
                _rec = well_images_real[_wid]
                _hw = pd.read_csv(well_hw_path(_wid), usecols=['TVT', 'TVT_input'])
                _n_hw = len(_hw); _be = np.linspace(0, _n_hw, W + 1).astype(int)
                _post_m = _hw['TVT_input'].isna().values
                _tp = _hw['TVT'].values[_post_m]; _ok = ~np.isnan(_tp)
                def _dec(_er_col):
                    _pc = _rec['tvt_lo'] + (_er_col + 0.5) * BIN_SIZE
                    _pf = np.zeros(_n_hw, dtype=np.float64)
                    for _c in range(W): _pf[_be[_c]:_be[_c+1]] = _pc[_c]
                    return _pf[_post_m][_ok]
                _p32, _p16 = _dec(_er32[_b]), _dec(_er16[_b])
                _res32[_wid] = _tp[_ok] - _p32
                _res16[_wid] = _tp[_ok] - _p16
                _pdiv.append(np.abs(_p16 - _p32))
    for _m in _models: del _m
    _models = []
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

_wids_fp = sorted(_res32)
_pool = lambda res: float(np.sqrt(np.mean(np.concatenate([res[w] for w in _wids_fp]) ** 2)))
_r32, _r16 = _pool(_res32), _pool(_res16)
_dv = np.concatenate(_pdiv)
print(f'\n=== accuracy (paired, {len(_wids_fp)} wells) ===')
print(f'  pooled RMSE  fp32 = {_r32:.4f}   amp = {_r16:.4f}   delta = {_r16 - _r32:+.5f} ft')
print(f'  (replication check: fp32 should match the §16 ph0.0 number at the same k/T)')
print(f'  per-row |pred_amp - pred_fp32|: mean {_dv.mean():.4f}  p95 {np.percentile(_dv,95):.4f}  '
      f'max {_dv.max():.4f} ft')
_dw = {w: float(np.sqrt(np.mean(_res16[w]**2)) - np.sqrt(np.mean(_res32[w]**2))) for w in _wids_fp}
_worst = sorted(_dw, key=lambda w: -abs(_dw[w]))[:5]
print('  largest per-well RMSE deltas: '
      + ', '.join(f'{w}:{_dw[w]:+.3f}' for w in _worst))
_w32 = {w: float(np.sqrt(np.mean(_res32[w]**2))) for w in _wids_fp}
_tail = sorted(_w32, key=lambda w: -_w32[w])[:max(1, len(_wids_fp)//10)]
print(f'  worst-decile wells (fp32): pooled fp32 '
      f'{np.sqrt(np.mean(np.concatenate([_res32[w] for w in _tail])**2)):.4f} vs amp '
      f'{np.sqrt(np.mean(np.concatenate([_res16[w] for w in _tail])**2)):.4f}')

_d = np.empty(N_BOOT_FP); _warr = np.array(_wids_fp, dtype=object)
for _i in range(N_BOOT_FP):
    _bs = _warr[_fprng.randint(0, len(_warr), len(_warr))]
    _a = np.sqrt(np.mean(np.concatenate([_res16[w] for w in _bs]) ** 2))
    _b2 = np.sqrt(np.mean(np.concatenate([_res32[w] for w in _bs]) ** 2))
    _d[_i] = _a - _b2
_lo_fp, _hi_fp = np.percentile(_d, [2.5, 97.5])
print(f'  paired bootstrap 95% CI (amp - fp32): [{_lo_fp:+.5f}, {_hi_fp:+.5f}]')

print(f'\n=== cost (this runtime\'s GPU — re-measure on Kaggle before betting the clock) ===')
print(f'  forward wall time: fp32 {_t32:.1f}s   amp {_t16:.1f}s   speedup x{_t32/max(_t16,1e-9):.2f}')
if _mem:
    print(f'  peak CUDA memory:  fp32 {_mem.get("fp32",0)/2**30:.2f} GiB   '
          f'amp {_mem.get("amp",0)/2**30:.2f} GiB')

print('\n=== VERDICT (pre-registered) ===')
_max_dw = max(abs(v) for v in _dw.values())
_ok1 = abs(_r16 - _r32) <= 0.010
_ok2 = float(np.percentile(_dv, 95)) <= 0.05
_ok3 = _max_dw <= 0.05
print(f'  |pooled delta| <= 0.010      : {"pass" if _ok1 else "FAIL"} ({abs(_r16-_r32):.5f})')
print(f'  divergence p95 <= 0.05 ft    : {"pass" if _ok2 else "FAIL"} ({np.percentile(_dv,95):.4f})')
print(f'  max |well RMSE delta| <= 0.05: {"pass" if _ok3 else "FAIL"} ({_max_dw:.4f})')
if _ok1 and _ok2 and _ok3:
    print('  -> SAFE: autocast is numerically inert on this member at deployed depth. If the')
    print('     measured speedup justifies it on Kaggle hardware, wire the per-member amp flag.')
else:
    print('  -> FLAG: half precision moves this member beyond the pre-registered envelope —')
    print('     do not ship fp16 for it; prefer dropping the member from TTA to fit runtime.')

In [ ]:
# ============================================================================
# §16 · PHASEJITTER TTA — per-phase pass + composition search (V2-L, k=PJ_K)
# Paste at the bottom of the OOF notebook (needs cells 1-11: fold_splits,
# well_images_real (with _raw), build_image_from_arrays, build_model, _dir_ckpts).
#
# CONTEXT: PhaseJitter TTA helps every model on the BOARD but taxes CV (§32.1
# −0.034 tax; §32.9: the board win is a real test-distribution effect CV cannot
# see). This cell prices the CV side at PHASE granularity on the current V2-L
# member AT ITS DEPLOYED DEPTH (PJ_K=3: handoff + refeed rounds, run per phase
# image — refeed re-reads each phase's own belief, so phase damage can compound
# with k in ways a k=1 probe understates). One pass renders all 8 phase variants
# per well (batched), keeps per-phase per-well residuals, then
# composes offline — singles, the deployed full-8 comb, prefix marginal-adds,
# rotated best subset (255), rotated free weights (exact simplex QP).
#
# INSTRUMENT: the notebook's own phase_h render (bin-edge shift + clip) — the
# §32-era harness whose equivalence to the kernel's toe-crop path was proven at
# 0.000e+00 (§32.1). With phase_h the uncovered gap sits at the HEEL (pre-PS
# rows), so post-PS rows are covered by EVERY phase (asserted below) and phase
# composition is exactly linear -> Gram machinery applies.
#
# WHAT THIS CANNOT SEE (pre-registered framing): the LB benefit is a test-
# distribution effect (§32.9) invisible on OOF by construction. The decision
# numbers here are CV-tax numbers:
#   T_full = full-8 comb minus ph0.0 (the tax the deployed TTA pays on this line)
#   T_sub  = rotated best-subset comb minus ph0.0
# READ: if T_full <= 0.015 ft, the tax is negligible — ship full TTA in the TTA
# slot without further thought. If a subset with >= 4 phases cuts >= 0.02 ft off
# T_full, it is a candidate reduced-tax TTA slot — but flag: subsetting on CV may
# also subset away the (unmechanized, 74% on-grid) board benefit, so prefer the
# full comb unless the tax is large.
# ============================================================================
import itertools as _itp

PJ_DIR     = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore/5fold_l2_stack2_ema_25000pt_mdwarp42_bedwarp42_handoffT15_expa_pt'
PJ_ENCODER = 'tu-tf_efficientnet_l2.ns_jft_in1k_475'
PJ_OFFSETS = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875]   # the kernel's full-bin sweep
PJ_FOLDS   = list(ALL_FOLDS)          # [0] for a quick single-fold look first
PJ_K       = 1                        # refeed depth — 3 = this member's DEPLOYED config
PJ_T       = 1.5                      # handoff T (this notebook's pinned eval)
PJ_REFEED_T = 1.0                     # refeed T for rounds 2..PJ_K
PJ_SAVE    = 'phase_tta_residuals_k3.npz'
N_BOOT_P   = 2000
_prng      = np.random.RandomState(4242)

assert '_dir_ckpts' in globals(), 'run the OOF driver cell first — needs _dir_ckpts'
assert os.path.isdir(PJ_DIR), f'not a directory: {PJ_DIR}'
ENCODER = PJ_ENCODER; DECODER_INTERP = 'nearest'
NP_ = len(PJ_OFFSETS)
print(f'PhaseJitter probe: k={PJ_K} T={PJ_T}/{PJ_REFEED_T}, {NP_} phases {PJ_OFFSETS}, folds={PJ_FOLDS}')
print('  cost/seed-well: 1 stage-1 + {0} stage-2 passes on an {1}-image phase batch'.format(PJ_K, NP_))
print('  NOTE: ph0.0 at the deployed k should reproduce this member\'s cached OOF standalone —')
print('        a free replication check against the driver run.\n')

# ---- 1 · the pass ----------------------------------------------------------------------
_P_res = {o: {} for o in PJ_OFFSETS}       # offset -> wid -> residual (post∧ok rows)
_uncov = 0
for _f in PJ_FOLDS:
    _cps = _dir_ckpts(PJ_DIR, _f)
    _models = []
    for _p in _cps:
        _m = build_model(encoder_name=PJ_ENCODER, in_ch=IN_CH).to(DEVICE)
        _m.load_state_dict(torch.load(_p, map_location=DEVICE)); _m.eval(); _models.append(_m)
    for _wid in tqdm(fold_splits[_f]['val'], desc=f'fold {_f} ({len(_cps)} seeds x {NP_} phases, k={PJ_K})'):
        _raw = well_images_real[_wid]['_raw']
        _recs = [build_image_from_arrays(**_raw, phase_h=float(_o)) for _o in PJ_OFFSETS]
        _imgs = torch.from_numpy(np.stack([r['image_in'] for r in _recs])).to(DEVICE)
        _er_sum = np.zeros((NP_, W), dtype=np.float64)
        with torch.no_grad():
            for _m in _models:
                _l1 = _m.stage1(_imgs).squeeze(1)
                _post = F.softmax(_l1 / float(PJ_T), dim=1)      # handoff (round 1)
                _l2 = _m._stage2_from_post(_imgs, _post)
                for _r in range(2, PJ_K + 1):                    # refeed rounds 2..k
                    _post = F.softmax(_l2 / float(PJ_REFEED_T), dim=1)
                    _l2 = _m._stage2_from_post(_imgs, _post)
                _er_sum += (F.softmax(_l2, dim=1)
                            * row_idx_t[None, :, None]).sum(dim=1).cpu().numpy()
        _er = _er_sum / len(_models)                          # seed-mean, per phase
        _hw = pd.read_csv(well_hw_path(_wid), usecols=['TVT', 'TVT_input'])
        _n_hw = len(_hw)
        _post_m = _hw['TVT_input'].isna().values
        _tp = _hw['TVT'].values[_post_m]; _ok = ~np.isnan(_tp)
        for _j, _o in enumerate(PJ_OFFSETS):
            _rec = _recs[_j]
            # build_image_from_arrays does not return bin_edges — recompute with its exact formula
            if float(_o) != 0.0:
                _be = np.clip(np.linspace(0, _n_hw, W + 1) + float(_o) * (_n_hw / W),
                              0, _n_hw).astype(int)
            else:
                _be = np.linspace(0, _n_hw, W + 1).astype(int)
            _pc = _rec['tvt_lo'] + (_er[_j] + 0.5) * BIN_SIZE
            _pf = np.full(_n_hw, np.nan, dtype=np.float64)
            for _c in range(W):
                if _be[_c + 1] > _be[_c]: _pf[_be[_c]:_be[_c + 1]] = _pc[_c]
            _pv = _pf[_post_m][_ok]
            if np.isnan(_pv).any():                            # should not happen (heel-gap only)
                _uncov += int(np.isnan(_pv).sum())
                _pv = pd.Series(_pv).ffill().bfill().values
            _P_res[_o][_wid] = (_tp[_ok] - _pv).astype(np.float64)
    for _m in _models: del _m
    _models = []
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
if _uncov: print(f'  WARNING: {_uncov} post-PS rows lacked phase coverage (ffilled) — '
                 f'linearity is approximate on those rows.')

_widsP = sorted(_P_res[PJ_OFFSETS[0]])
_foldsP = {w: f for f in PJ_FOLDS for w in fold_splits[f]['val']}
_GP, _NPr = {}, {}
for _w in _widsP:
    _E = np.stack([_P_res[o][_w] for o in PJ_OFFSETS], axis=1)
    _GP[_w] = _E.T @ _E; _NPr[_w] = _E.shape[0]
_totP = sum(_NPr.values())
if PJ_SAVE:
    _pack = {}
    for _j, _o in enumerate(PJ_OFFSETS):
        _a = np.empty(len(_widsP), dtype=object)
        for _i, _w in enumerate(_widsP): _a[_i] = _P_res[_o][_w].astype(np.float32)
        _pack[f'res_P{_j}'] = _a
    np.savez(PJ_SAVE, well_ids=np.array(_widsP),
             offsets=np.array(PJ_OFFSETS, dtype=np.float64), **_pack)
    print(f'per-phase residuals saved -> {PJ_SAVE}')

def _opt_qpP(_G):
    """Exact min of w^T G w on the simplex (active-set enumeration, any dimension)."""
    _m = _G.shape[0]; _bw, _bv = None, np.inf
    for _r in range(1, _m + 1):
        for _S in _itp.combinations(range(_m), _r):
            _GS = _G[np.ix_(_S, _S)]
            try: _x = np.linalg.lstsq(_GS, np.ones(_r), rcond=None)[0]
            except np.linalg.LinAlgError: continue
            _s = _x.sum()
            if not np.isfinite(_s) or abs(_s) < 1e-12: continue
            _x = _x / _s
            if (_x < -1e-9).any(): continue
            _x = np.clip(_x, 0.0, None); _x = _x / _x.sum()
            _wv = np.zeros(_m); _wv[list(_S)] = _x
            _v = float(_wv @ _G @ _wv)
            if _v < _bv: _bv, _bw = _v, _wv
    return _bw
def _e_vecP(idx):
    _v = np.zeros(NP_); _v[list(idx)] = 1.0 / len(idx); return _v
def _rmseP(wv, wells=None):
    _ws = wells if wells is not None else _widsP
    return float(np.sqrt(sum(float(wv @ _GP[u] @ wv) for u in _ws) / sum(_NPr[u] for u in _ws)))

# ---- 2 · per-phase standalones + mechanism profile -------------------------------------
_toe_sse = {o: 0.0 for o in PJ_OFFSETS}; _toe_n = 0
for _w in _widsP:
    _n = _NPr[_w]; _cut = int(np.floor(0.90 * _n))
    _toe_n += _n - _cut
    for _o in PJ_OFFSETS: _toe_sse[_o] += float(np.sum(_P_res[_o][_w][_cut:] ** 2))
print(f'\n=== per-phase standalone (k={PJ_K}, seed-ensembled, {len(_widsP)} wells) ===')
print('  phase    RMSE      d vs ph0   toe(90-100%)  d-toe')
_singlesP = {o: _rmseP(_e_vecP([j])) for j, o in enumerate(PJ_OFFSETS)}
_toe0 = np.sqrt(_toe_sse[PJ_OFFSETS[0]] / _toe_n)
for _o in PJ_OFFSETS:
    _t = np.sqrt(_toe_sse[_o] / _toe_n)
    print(f'  {_o:5.3f}  {_singlesP[_o]:8.4f}  {_singlesP[_o]-_singlesP[0.0]:+8.4f}'
          f'   {_t:10.4f}  {_t-_toe0:+7.4f}')

print('\n=== phase-residual correlation matrix (lower = all rows | upper = worst decile) ===')
_flatP = {o: np.concatenate([_P_res[o][w] for w in _widsP]) for o in PJ_OFFSETS}
_alp = {o: f'P{j}' for j, o in enumerate(PJ_OFFSETS)}
print('        ' + ' '.join(f'{_alp[o]:>6s}' for o in PJ_OFFSETS)
      + '     (' + ', '.join(f'{_alp[o]}={o}' for o in PJ_OFFSETS) + ')')
for _i, _oa in enumerate(PJ_OFFSETS):
    _cells = []
    for _j, _ob in enumerate(PJ_OFFSETS):
        if _j == _i: _cells.append(f'{"·":>6s}'); continue
        if _j > _i:
            _comb = np.abs(_flatP[_oa]) + np.abs(_flatP[_ob])
            _wd = _comb >= np.quantile(_comb, 0.90)
            _cells.append(f'{np.corrcoef(_flatP[_oa][_wd], _flatP[_ob][_wd])[0,1]:6.3f}')
        else:
            _cells.append(f'{np.corrcoef(_flatP[_oa], _flatP[_ob])[0,1]:6.3f}')
    print(f'  {_alp[_oa]:>4s}  ' + ' '.join(_cells))

# ---- 3 · compositions ------------------------------------------------------------------
_full = _e_vecP(range(NP_))
_r_full = _rmseP(_full)
print(f'\n=== compositions (baseline = ph0.0 single pass, {_singlesP[0.0]:.4f}) ===')
print(f'  FULL-8 equal comb (the deployed TTA object): {_r_full:.4f}   '
      f'CV TAX = {_r_full - _singlesP[0.0]:+.4f} ft')
print('  prefix marginal-adds (comb of first m offsets; the "which phase hurts" curve):')
_prev = _singlesP[0.0]
for _m2 in range(2, NP_ + 1):
    _r = _rmseP(_e_vecP(range(_m2)))
    print(f'    + ph{PJ_OFFSETS[_m2-1]:5.3f} -> comb({_m2}) = {_r:.4f}   marginal {_r-_prev:+.4f}')
    _prev = _r

# rotated best subset (255) + rotated free weights
_subsP = [s for r in range(1, NP_ + 1) for s in _itp.combinations(range(NP_), r)]
def _fit_subP(wells):
    _Gs = sum(_GP[u] for u in wells)
    return min((_e_vecP(s) for s in _subsP), key=lambda g: float(g @ _Gs @ g))
def _fit_freeP(wells):
    return _opt_qpP(sum(_GP[u] for u in wells))
_appl = {'rot-subset': {}, 'rot-weights': {}, 'full8': {}, 'ph0': {}}
for _f0 in PJ_FOLDS:
    _wfit = [w for w in _widsP if _foldsP[w] != _f0]
    _wapp = [w for w in _widsP if _foldsP[w] == _f0]
    if not _wfit or not _wapp: continue
    _ws = _fit_subP(_wfit); _wf = _fit_freeP(_wfit)
    for _w in _wapp:
        _appl['rot-subset'][_w] = _ws; _appl['rot-weights'][_w] = _wf
        _appl['full8'][_w] = _full; _appl['ph0'][_w] = _e_vecP([0])
_rotP = {}
for _v in _appl:
    _sse = sum(float(_appl[_v][w] @ _GP[w] @ _appl[_v][w]) for w in _widsP)
    _rotP[_v] = float(np.sqrt(_sse / _totP))
print(f"\n=== fold-rotated (the fair numbers) ===")
print(f"  ph0.0 single           : {_rotP['ph0']:.4f}")
print(f"  full-8 comb            : {_rotP['full8']:.4f}   TAX {_rotP['full8']-_rotP['ph0']:+.4f}")
print(f"  rot best subset        : {_rotP['rot-subset']:.4f}   TAX {_rotP['rot-subset']-_rotP['ph0']:+.4f}"
      f"   (last-rotation subset: {[PJ_OFFSETS[j] for j in range(NP_) if _ws[j] > 0]})")
print(f"  rot free weights       : {_rotP['rot-weights']:.4f}   TAX {_rotP['rot-weights']-_rotP['ph0']:+.4f}"
      f"   (last-rotation w: {[round(float(x),2) for x in _wf]})")

def _bootP(a, b):
    _d = np.empty(N_BOOT_P); _arr = np.array(_widsP, dtype=object)
    for _i in range(N_BOOT_P):
        _bs = _arr[_prng.randint(0, len(_arr), len(_arr))]
        _nn = sum(_NPr[w] for w in _bs)
        _ra = np.sqrt(sum(float(_appl[a][w] @ _GP[w] @ _appl[a][w]) for w in _bs) / _nn)
        _rb = np.sqrt(sum(float(_appl[b][w] @ _GP[w] @ _appl[b][w]) for w in _bs) / _nn)
        _d[_i] = _ra - _rb
    return np.percentile(_d, [2.5, 97.5])
_ci_full = _bootP('full8', 'ph0'); _ci_sub = _bootP('rot-subset', 'ph0')
print(f"  bootstrap 95% CI: full-8 tax [{_ci_full[0]:+.4f}, {_ci_full[1]:+.4f}]   "
      f"subset tax [{_ci_sub[0]:+.4f}, {_ci_sub[1]:+.4f}]")

# ---- 4 · read-out (pre-registered framing in the header) -------------------------------
print('\n=== READ-OUT ===')
_tax = _rotP['full8'] - _rotP['ph0']
if _tax <= 0.015:
    print(f'  full-8 CV tax {_tax:+.4f} <= 0.015: NEGLIGIBLE on this line — ship the full TTA')
    print('  in the TTA slot; the board benefit (§32.9) rides at ~no CV cost here.')
else:
    _nsub = int(sum(_ws > 0))
    _cut = _rotP['full8'] - _rotP['rot-subset']
    if _nsub >= 4 and _cut >= 0.02:
        print(f'  full-8 tax {_tax:+.4f}; the rotated {_nsub}-phase subset cuts {_cut:.3f} ft of it.')
        print('  CANDIDATE reduced-tax TTA slot — but the board benefit is unmechanized (74%')
        print('  on-grid strand, §32.9), so subsetting on CV may subset the benefit away too.')
        print('  Prefer the full comb unless the tax materially outweighs the ~0.26 board delta.')
    else:
        print(f'  full-8 tax {_tax:+.4f} with no admissible cheaper subset (>=4 phases, >=0.02')
        print('  saved). The choice stays binary: slot 1 no-TTA (CV bet) vs slot 2 full TTA')
        print('  (test-distribution bet) — exactly the §32.9.7 slot logic, now priced on this line.')